# Proactive Churn - v1 dataset extract

Connects to the brightwheel Analytics Redshift cluster and builds the v1 modelling dataset. This
notebook is *extract only*: it produces the full candidate feature set, and feature selection
(BorutaSHAP, the billing ablation, the leakage check) and modelling are downstream of it.

**What it produces**, in `data/raw/` (gitignored):

| file | contents | audience |
| --- | --- | --- |
| `churn_v1_..._h3.parquet` | the canonical readable dataset - one row per school x month, categoricals as `category` dtype | audits, rep-facing output |
| `..._h3_encoded.parquet` | the same rows with the categorical features one-hot encoded - the model matrix | model fitting |
| `..._h3_encoding.json` | the encoding schema: `model_cols` in exact order plus the train levels each dummy came from | a scoring job rebuilding the matrix |

**Sections**

1. Imports
2. Credentials and paths
3. Connection helpers
4. Smoke test
5. Project tables (5a. catalog snapshot)
6. Configuration - every tunable constant
7. The candidate feature set (7a. reconcile against the catalog, 7b. guardrails)
8. The extract
9. The 2x2 school-by-time split
10. Save
11. Encode the categoricals for the model matrix
12. Appendix. Data checks - optional

**Before you run anything:**

1. **Be on VPN.** The cluster is not reachable off-VPN - you get a connection timeout, not an auth error. ([VPN setup guide](https://docs.google.com/document/d/1kNqdf4H1voZC0ne0UI1oH-K66tn8EBZ0OfC-Ivh-wUc/edit))
2. **Fill in `.env`** at the project root (`cp .env.example .env`). Credentials live there, never in this notebook.
3. **Kernel = "Python (Proactive Churn)"** - created by `setup.sh`.

Design decisions and the rationale behind them live in the Obsidian vault at
`Proactive Churn/Design Decisions.md`.

## 1. Imports

Connection and extract only - no modelling libraries.

In [ ]:
import json
import os
import re
import time
from contextlib import contextmanager
from pathlib import Path

import numpy as np
import pandas as pd

import redshift_connector
from dotenv import load_dotenv

## 2. Credentials and paths

Loads the gitignored `.env` at the project root and creates `RAW_DIR = data/raw/`, which is also
gitignored - school-level customer data must not land in git. Nothing secret is printed.

Credentials are issued by Data Eng on the `bw_analytics` database; password rotation steps are in
`README.md`.

In [ ]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "data" else Path.cwd()
DOTENV_PATH = PROJECT_ROOT / ".env"

if not DOTENV_PATH.exists():
    raise FileNotFoundError(
        f"No .env at {DOTENV_PATH}. Run: cp .env.example .env, then fill in REDSHIFT_PASSWORD"
    )

load_dotenv(DOTENV_PATH, override=True)

REDSHIFT_CONFIG = {
    "host": os.environ["REDSHIFT_HOST"],
    "port": int(os.environ.get("REDSHIFT_PORT", 5439)),
    "database": os.environ["REDSHIFT_DATABASE"],
    "user": os.environ["REDSHIFT_USER"],
    "password": os.environ["REDSHIFT_PASSWORD"],
    # A real redshift_connector.connect() keyword. WITHOUT it, DECIMAL / NUMERIC
    # columns arrive as decimal.Decimal objects in OBJECT-dtype columns, where
    # they bypass any dtype-based numeric handling downstream. §10 no longer
    # relies on dtype inference either - see its numeric-set-by-exclusion note.
    "numeric_to_float": True,
}

_missing = [k for k, v in REDSHIFT_CONFIG.items() if v in (None, "")]
if _missing:
    raise ValueError(f"Empty value(s) in .env: {_missing}")

DATA_DIR = PROJECT_ROOT / "data"
RAW_DIR = DATA_DIR / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Echo everything EXCEPT the password.
print({k: v for k, v in REDSHIFT_CONFIG.items() if k != "password"})
print(f"password: <set, {len(REDSHIFT_CONFIG['password'])} chars>")
print(f"extracts -> {RAW_DIR}")

## 3. Connection helpers

Defines three things:

- `redshift_connection(**overrides)` - a context manager giving one connection per query, always
  closed. A connection left open across a long notebook session goes stale and the next cell fails
  with an opaque socket error.
- `run_query(sql, params)` - returns a DataFrame via `cursor.fetch_dataframe()`. Read-only: it
  blocks anything that is not a `SELECT` / `WITH` / `SHOW` / `EXPLAIN`.
- `execute_statement(sql, params)` - for the rare write. Commits.

In [ ]:
@contextmanager
def redshift_connection(**overrides):
    """Open a Redshift connection, guaranteed closed on exit."""
    conn = redshift_connector.connect(**{**REDSHIFT_CONFIG, **overrides})
    try:
        yield conn
    finally:
        conn.close()


_READ_ONLY_PREFIXES = ("select", "with", "show", "explain")


def run_query(sql: str, params=None) -> pd.DataFrame:
    """Run a read-only query and return a DataFrame.

    Parameters
    ----------
    sql : str
        A SELECT / WITH / SHOW / EXPLAIN statement.
    params : sequence, optional
        Values for %s placeholders. Use these instead of f-strings - string
        interpolation into SQL is how you end up with an injection bug or a
        silently wrong filter.
    """
    stripped = sql.strip().lstrip("(").lower()
    if not stripped.startswith(_READ_ONLY_PREFIXES):
        raise ValueError("run_query is read-only. Use execute_statement() for DDL/DML.")

    with redshift_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            df = cur.fetch_dataframe()

    # fetch_dataframe returns None for a zero-row result set.
    return df if df is not None else pd.DataFrame()


def execute_statement(sql: str, params=None) -> None:
    """Run a statement that returns no rows (DDL, ALTER USER, ...). Commits."""
    with redshift_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(sql, params)
        conn.commit()


print("helpers defined: redshift_connection, run_query, execute_statement")

## 4. Smoke test

First thing to run. If this returns a row, the connection is good.

| Symptom | Likely cause |
| --- | --- |
| Hangs, then timeout / "could not translate host name" | Not on VPN |
| `password authentication failed` | Wrong `REDSHIFT_PASSWORD` in `.env` (or you rotated it and didn't update `.env`) |
| `permission denied for relation ...` | Connection is fine; you need a GRANT - ask in `#ask-dataeng` |

In [ ]:
run_query(
    """
    select
        current_user       as connected_as
      , current_database() as db
      , getdate()          as server_time_utc
      , version()          as redshift_version
    """
)

## 5. Project tables

The four tables this notebook reads, named once in `TABLES` and referenced from there everywhere.

| Table | Grain | Role in v1 |
| --- | --- | --- |
| `rpt_monthly_school_usage_trends` | school x month | **The inputs.** Every candidate feature. |
| `rpt_school_churn_cases` | one row per retention event | **The answer.** Cancellation requests. |
| `mart_monthly_school_customer_status` (MMSC) | school x month | **Billing-attach history** - the six columns behind `has_ever_billed`. |
| `mart_school_customers` | school (current state) | Contact list for outreach. **Not used to train.** |

Only the six billing-attach columns are taken from MMSC; §7 lists them as `MMSC_FEATURES` and §8
joins them in. To check that you have `SELECT` on all four and see their row counts, run §12a.

In [ ]:
SCHEMA = "analytics_mart_customer"

TABLES = {
    "usage_trends":    f"{SCHEMA}.rpt_monthly_school_usage_trends",           # inputs
    "churn_cases":     f"{SCHEMA}.rpt_school_churn_cases",                    # label
    "billing_history": f"{SCHEMA}.mart_monthly_school_customer_status",       # billing attach (MMSC)
    "customers":       f"{SCHEMA}.mart_school_customers",                     # outreach only
}

TABLES

### 5a. Catalog snapshot

Reads the usage-trends column list from `information_schema` into `USAGE_COLS`. §7a generates the
spec's feature names by naming convention and intersects them with this set, so a renamed or missing
warehouse column shows up as a printed gap rather than a SQL error 40 columns into a 200-column
select. §7 also probes this set for an organization identifier.

In [ ]:
def describe_table(table: str) -> pd.DataFrame:
    """Column list for a table, via svv_columns with a zero-row probe fallback.

    NOT information_schema.columns. In Redshift that view returns ZERO ROWS for a
    LATE-BINDING VIEW, and the rpt_ models are built that way - so it reports the
    table as having no columns while SELECT against the same table works fine.
    svv_columns does cover late-binding views; the `select * limit 0` probe below
    is the belt to that braces, since it needs nothing but SELECT privilege and
    works for tables, views, late-binding views and external tables alike.
    """
    schema, name = table.split(".", 1)

    df = run_query(
        """
        select
            ordinal_position as pos
          , column_name
          , data_type
          , is_nullable
        from svv_columns
        where table_schema = %s
          and table_name   = %s
        order by ordinal_position
        """,
        params=(schema, name),
    )
    if len(df):
        return df

    # Fallback: ask the table itself. cursor.description is populated even when
    # every metadata view is blind to the object.
    print(f"  svv_columns returned nothing for {table} - falling back to a "
          "zero-row probe")
    with redshift_connection() as conn:
        with conn.cursor() as cur:
            cur.execute(f"select * from {table} limit 0")
            names = [d[0].decode() if isinstance(d[0], bytes) else d[0]
                     for d in cur.description]
    return pd.DataFrame({
        "pos": range(1, len(names) + 1),
        "column_name": names,
        "data_type": "(from cursor probe)",
        "is_nullable": "(unknown)",
    })


USAGE_CATALOG = describe_table(TABLES["usage_trends"])
USAGE_COLS = set(USAGE_CATALOG["column_name"])

# FLOOR GUARD. A blind metadata view does not raise - it returns an empty result,
# and §7a then keeps only the generated names that "exist", which is none of
# them. The extract still succeeds and still writes a parquet, containing keys,
# label and the MMSC columns that bypass this catalog. That is a silent 11-column
# dataset that looks like a successful run. Fail here instead.
MIN_EXPECTED_CATALOG_COLS = 150   # the model documents ~212

assert len(USAGE_COLS) >= MIN_EXPECTED_CATALOG_COLS, (
    f"only {len(USAGE_COLS)} column(s) found for {TABLES['usage_trends']}, expected "
    f"at least {MIN_EXPECTED_CATALOG_COLS}. The catalog probe is not seeing the "
    "table - almost certainly a permissions gap on svv_columns, or the fallback "
    "probe failed. Do NOT proceed: every trends-sourced feature would be dropped "
    "as 'not in the warehouse' and the extract would still write a parquet."
)

print(f"{len(USAGE_COLS):,} columns in {TABLES['usage_trends']}")
USAGE_CATALOG.head(15)

---

## 6. Configuration

Every tunable constant in one cell: the observation window, the label definition, the population
filters and the split constants. Change one here and re-run from §8 to get a new dataset - there is
no SQL to edit.

Derived values are computed from those constants rather than typed: `LAST_ANSWERABLE_MONTH` and
`OBS_END` from `AS_OF_DATE` + `LABEL_HORIZON_MONTHS`, and `EVAL_START` (the first `TEST_YEAR` month
§9 is willing to evaluate) from `TRAIN_YEAR` + `LABEL_HORIZON_MONTHS`. `EVAL_START` is asserted
against its expected value, so changing the horizon cannot silently move the split boundary.

`norm_country` is defined here too: it normalises a `school_country` value the same way the SQL
predicate does, and §12b uses it.

The long feature-family lists are **not** here - they are a transcription of the spec, in §7.

In [ ]:
OBS_START = "2024-01-01"     # inclusive
AS_OF_DATE = "2026-09-01"    # pinned. Bump deliberately to refresh the dataset.
LABEL_HORIZON_MONTHS = 3
# --- label -----------------------------------------------------------------
# The SQL wraps the flag as coalesce(REQUEST_FLAG, false) - it is nullable on
# some models, and a bare predicate would drop NULL rows via three-valued logic.
REQUEST_FLAG = "is_reportable_request"
CANCEL_REQUEST_TYPES = (
    "Addressable Cancelation Request",
    "Non-Addressable Cancelation Request",
)
EXCLUDED_REASON_CATEGORIES = (
    "Failed Payment",
    "Business Changes",
    "Facility Closed",
)
APPLY_TARGET_POPULATION = True   # adds `and is_target_population` to the events CTE

# --- population ------------------------------------------------------------
PREMIUM_FILTER = "is_premium_sub_or_invoice"   # raw SQL predicate: paying SaaS this month
GOV_FILTER     = "not is_gov"                  # raw SQL predicate: non-government
POPULATION_SEGMENTS = (
    "10-39 small",
    "40-99 medium",
    "100+ large",
    "SMB Multisite 2-9",
)
# Excluded: "< 10 micro" and "Enterprise 10+".

# The SQL compares upper(trim(school_country)), so every entry here must already
# be upper-cased and trimmed. §12b prints the stored values and marks anything
# outside this pair as DROP.
POPULATION_COUNTRIES = ("UNITED STATES", "CANADA")

# --- split -----------------------------------------------------------------
TRAIN_YEAR = 2024
TEST_YEAR  = 2025
SCHOOL_SPLIT_SEED     = 42
TRAIN_SCHOOL_FRACTION = 0.70   # share of schools in group A

# --- derived ---------------------------------------------------------------
# A month is "answerable" only once its whole forward window has closed, i.e.
# once month + HORIZON is a fully elapsed month. The as-of month is not elapsed,
# so that is HORIZON + 1 months back from the start of it.
_this_month = pd.Timestamp(AS_OF_DATE).normalize().replace(day=1)
LAST_ANSWERABLE_MONTH = _this_month - pd.DateOffset(months=LABEL_HORIZON_MONTHS + 1)
OBS_END = (LAST_ANSWERABLE_MONTH + pd.DateOffset(months=1)).strftime("%Y-%m-%d")  # exclusive

# The last training month is TRAIN_YEAR-12 and its label window reaches
# LABEL_HORIZON_MONTHS forward, so the first TEST_YEAR month whose OWN label
# window starts after every training window has closed is
# TRAIN_YEAR-12 + (HORIZON + 1) months.
EVAL_START = (pd.Timestamp(year=TRAIN_YEAR, month=12, day=1)
              + pd.DateOffset(months=LABEL_HORIZON_MONTHS + 1))
assert EVAL_START == pd.Timestamp("2025-04-01"), (
    f"EVAL_START derived as {EVAL_START:%Y-%m-%d}, expected 2025-04-01 - the split "
    "boundary moved, almost certainly because LABEL_HORIZON_MONTHS changed. It "
    "re-cuts embargo / holdout_time / test, so re-check the boundary and update "
    "this assertion deliberately."
)

def norm_country(v):
    """Normalise a school_country value the same way the SQL predicate does.

    upper(trim(...)) in Redshift; NULL/NaN passes through unchanged so it never
    compares equal to an accepted value.
    """
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return v
    return str(v).strip().upper()


print(f"as of      {AS_OF_DATE} (pinned)")
print(f"window     {OBS_START} -> {OBS_END} (exclusive)")
print(f"           last answerable observation month: {LAST_ANSWERABLE_MONTH:%Y-%m}")
print(f"label      coalesce({REQUEST_FLAG}, false) + summarized_request_type in "
      f"{CANCEL_REQUEST_TYPES} + case_created_date is not null")
print(f"horizon    {LABEL_HORIZON_MONTHS} months forward (one `label` column)")
print(f"target pop {'APPLIED' if APPLY_TARGET_POPULATION else 'SKIPPED'}")
print(f"segments   {POPULATION_SEGMENTS}")
print(f"countries  {POPULATION_COUNTRIES}  (compared as upper(trim(school_country)))")
print(f"split      schools {TRAIN_SCHOOL_FRACTION:.0%} A / {1 - TRAIN_SCHOOL_FRACTION:.0%} B "
      f"(seed {SCHOOL_SPLIT_SEED}, stratified on ever-positive)")
print(f"           x time: train {TRAIN_YEAR} / test {TEST_YEAR} from "
      f"{EVAL_START:%Y-%m} (derived)")

---

## 7. The candidate feature set

A transcription of the spec's feature list, family by family. Nothing here is a knob - the tunables
are in §6.

For the trend-eligible families the spec asks for four versions of each base signal - the current
month, the trailing-3-month average, the change vs. that average, and the change vs. a year ago.
Those are generated from the naming convention in §7a rather than typed out, then intersected with
`USAGE_COLS`.

The cell below also defines:

- `EXCLUDE_EXACT` and `EXCLUDE_PATTERNS` - names that must never become inputs;
- `MMSC_FEATURES` - the six billing-attach columns, which come from MMSC rather than usage-trends
  and are therefore appended to `FEATURES` *after* the catalog reconciliation;
- `ORG_COL` - an organization identifier probed from the catalog and carried as a non-feature
  passthrough for §12e's leakage diagnostic.

In [ ]:
# ---------------------------------------------------------------------------
# Base signals, transcribed from column_selection_spec.md, "What goes into the
# model".
# ---------------------------------------------------------------------------

# Money / payments.
MONEY_BASE = [
    "billing_net_gpv_local",
    "billing_net_gpv_local_card",
    "students_with_online_payment",
    "school_payer_attach_rate",
    "product_active_students",
    "students_with_payment_method_added",
    "num_billing_invoices_due",
    "billing_card_fee_percentage",
    "billing_ach_fee_percentage",
    "pct_invoices_paid_online_proxy",
]

# Product use - the 13 "did they use it at all" flags.
USAGE_FLAG_BASE = [
    "any_messaging",
    "any_admission_funnel",
    "any_online_payment",
    "any_activity",
    "any_timecard",
    "any_attendance",
    "any_form",
    "any_ec_lesson_plan",
    "any_calendar",
    "any_payroll",
    "any_expense",
    "any_report_builder",
    "any_time_off",
]

# Product use - the 9 "how much per student" intensity rates. Resolved from the
# catalog by suffix in §7a rather than listed, because the spec names one
# ("messaging_events_per_active_student") and says "and its siblings".
PER_STUDENT_SUFFIX = "_per_active_student"

# Warning signs - billing-plan stoppages.
WARNING_BASE = [
    "negative_reason_count",
    "reason_school_temp_paused_count",
    "reason_school_temp_reducing_enrollment_count",
    "reason_school_permanently_closing_count",
    "reason_school_no_longer_using_bw_count",
]

# Simple current-state fields - no trend variants.
CURRENT_STATE = [
    "num_billing_invoices_created",
    "num_students_with_billing_invoice_created",
    "num_billing_plans_created",
    "num_students_with_billing_plan_created",
    "has_cc_processing_fees_enabled_binary",
    "has_ach_processing_fees_enabled_binary",
    "num_fees_passed_to_student",
    "enforces_autopay_binary",
    "prev_month_enforces_autopay_binary",
    "delta_enforces_autopay_binary",
]

# Past history.
HISTORY = [
    "months_since_last_cancel_request",
    "running_saas_cancel_request_count",
    "months_since_last_failed_payment_case",
    "running_saas_failed_payment_count",
]

# Descriptive.
DESCRIPTIVE = [
    "customer_segment",
    "monthly_billing_status",
    "school_country",
]

# ---------------------------------------------------------------------------
# Billing history, sourced from MMSC (mart_monthly_school_customer_status), NOT
# from usage-trends.
#
# CRITICAL: these names are NOT in USAGE_COLS, so they must never go through
# _expand() / _flat(), both of which drop anything absent from the usage-trends
# catalog. They are appended to FEATURES *after* the catalog reconciliation in
# §7a, and §7b scopes its "every feature is in USAGE_COLS" assertion to the
# trends-sourced features only. They are validated instead by §8's join, which
# counts the observation rows that got no MMSC match.
MMSC_FEATURES = [
    "months_since_first_processed_dollars_local",  # as of dim_month; null if never processed
    "processed_dollars_rank_local",                # 1 = first-ever processed month; null in months with none
    "has_processed_dollars_this_month_local",
    "has_processed_dollars_last_month_local",
    "monthly_billing_status_local",                # 7 documented lifecycle values - categorical
    "has_ever_billed",                             # derived in SQL from months_since_first_processed_dollars_local
]

# The one categorical among them. Documented values:
#   '1. New', '2. Retained', '3. Resurrected', '4. New Churn',
#   '5. Billing Inactive', '6. Never Billing', '7. Other'
MMSC_CATEGORICAL = ["monthly_billing_status_local"]

# ---------------------------------------------------------------------------
# Organization identifier - a NON-FEATURE passthrough, like product_score.
# The name is not documented, so probe the catalog for the plausible spellings
# in order and take the first that exists. If none does, ORG_COL stays None,
# §12e prints a warning and its diagnostic goes unmeasured - it never fails the
# run.
ORG_CANDIDATES = [
    "organization_id",
    "org_id",
    "brightwheel_organization_id",
    "organization_uuid",
    "org_uuid",
]
ORG_COL = next((c for c in ORG_CANDIDATES if c in USAGE_COLS), None)
if ORG_COL:
    print(f"org identifier   {ORG_COL!r} found in the usage-trends catalog "
          "(passthrough only, never a feature)")
else:
    print("org identifier   NONE of "
          f"{ORG_CANDIDATES} exists in the usage-trends catalog "
          "-> §12e's multisite-straddle diagnostic cannot run")

# ---------------------------------------------------------------------------
# The four versions of each trend-eligible signal.
# ---------------------------------------------------------------------------
VARIANT_TEMPLATES = [
    "{c}",                   # this month's value
    "trailing_3m_{c}",       # average of the last 3 months
    "delta_{c}_vs_t3m",      # change vs. that 3-month average
    "delta_{c}_vs_12m_ago",  # change vs. a year ago
]

# ---------------------------------------------------------------------------
# Exclusions. Spec: "Not used as inputs".
# ---------------------------------------------------------------------------
EXCLUDE_EXACT = {
    # The retention health score and its variants. product_score rides along in
    # PASSTHROUGH as the baseline (§7b), never as a feature.
    "product_score",
    "trailing_3m_product_score",
    "delta_product_score_vs_t3m",
    "delta_product_score_vs_12m_ago",
    # Fill-rate drift: 2% -> 64% across the split.
    "months_since_last_dm_connect",
    # Constant once the population is filtered to non-government schools.
    "is_gov",
    # Derived from the current-state SFDC dates. The dates themselves are caught
    # by EXCLUDE_PATTERNS below; these are the derived columns, which the pattern
    # cannot see.
    "months_since_billing_activation",
    "was_ever_billing_customer",
    "months_since_last_processed_dollars",   # MMSC's ..._local is used instead
}

# The org id is §12e's diagnostic key, never an input. Excluding it here means it
# cannot reach FEATURES by any route.
if ORG_COL:
    EXCLUDE_EXACT.add(ORG_COL)

EXCLUDE_PATTERNS = [
    # EXPIRED SIGNAL. The fee-percentage family separated cancellers from
    # non-cancellers inside a 2024 month at an AUC of ~0.85, and inside a 2025
    # month at ~0.50 - nothing. Card and ACH give nearly identical numbers, so
    # both appear to encode the same underlying pricing fact. The 2024 model was
    # dominated by them (SHAP 12.7 against 3.0 for second place) and scored 14x
    # over base on new schools in the same year, then BELOW base on a later
    # year. A relationship that no longer exists is worse than no feature: it
    # crowds out the ones that still transfer. Pattern, not exact names, so the
    # level, trailing_3m and both delta variants all go - 8 columns.
    re.compile(r"fee_percentage"),
    # Forward-looking: is_premium_sub_or_invoice_in_1m / _3m / _6m / _12m. A
    # pattern, so a newly added forward-looking column cannot slip in.
    re.compile(r"_in_(1|3|6|12)m$"),
    # Current-state Salesforce dates replicated across every historical row:
    # billing_activation_date_sfdc, saas_activation_date_sfdc, and anything
    # named the same way that gets added later.
    re.compile(r"_date_sfdc$"),
]

print(f"money            {len(MONEY_BASE):>3} base signals")
print(f"usage flags      {len(USAGE_FLAG_BASE):>3} base signals")
print(f"warning signs    {len(WARNING_BASE):>3} base signals")
print(f"current state    {len(CURRENT_STATE):>3} (no trend versions)")
print(f"history          {len(HISTORY):>3}")
print(f"descriptive      {len(DESCRIPTIVE):>3}")
print(f"billing (MMSC)   {len(MMSC_FEATURES):>3} (joined in, not from the usage-trends catalog)")

### 7a. Expand the variants and reconcile against the catalog

Builds `FEATURES` family by family: every name the spec implies is generated, each is kept only if
`USAGE_COLS` actually has it, and everything dropped is printed. The `MMSC_FEATURES` are appended
afterwards, and `TRENDS_FEATURES` is frozen before that append so §7b can scope its catalog
assertion to the trends-sourced names.

A family showing `not_in_warehouse` is worth a look - either the warehouse genuinely doesn't carry
that variant (a binary flag with no `_vs_12m_ago` version, say) or a column has been renamed.

In [ ]:
def _is_excluded(col: str) -> bool:
    return col in EXCLUDE_EXACT or any(p.search(col) for p in EXCLUDE_PATTERNS)


def _expand(bases):
    """Generate the four variants of each base signal; split into found / missing."""
    found, missing = [], []
    for b in bases:
        for tpl in VARIANT_TEMPLATES:
            name = tpl.format(c=b)
            (found if name in USAGE_COLS else missing).append(name)
    return found, missing


def _flat(cols):
    """Take columns as-is; split into found / missing."""
    return ([c for c in cols if c in USAGE_COLS],
            [c for c in cols if c not in USAGE_COLS])


# The 9 per-student rates, resolved from the catalog. Exclude the derived
# variants - _expand regenerates those.
PER_STUDENT_BASE = sorted(
    c for c in USAGE_COLS
    if c.endswith(PER_STUDENT_SUFFIX)
    and not c.startswith("trailing_3m_")
    and not c.startswith("delta_")
)
if len(PER_STUDENT_BASE) != 9:
    print(
        f"NOTE: spec says 9 per-student rates, catalog has {len(PER_STUDENT_BASE)}:\n"
        f"      {PER_STUDENT_BASE}\n"
        "      Reconcile with the spec before trusting the feature count.\n"
    )

FAMILIES = {
    "money (x4 variants)":         _expand(MONEY_BASE),
    "usage flags (x4 variants)":   _expand(USAGE_FLAG_BASE),
    "per-student (x4 variants)":   _expand(PER_STUDENT_BASE),
    "warning signs (x4 variants)": _expand(WARNING_BASE),
    "current state":               _flat(CURRENT_STATE),
    "history":                     _flat(HISTORY),
    "descriptive":                 _flat(DESCRIPTIVE),
}

FEATURES, _dropped_by_exclusion = [], []
for _family, (_found, _missing) in FAMILIES.items():
    for c in _found:
        if _is_excluded(c):
            _dropped_by_exclusion.append(c)
        elif c not in FEATURES:
            FEATURES.append(c)

report = pd.DataFrame([
    {
        "family": f,
        "requested": len(found) + len(missing),
        "kept": len([c for c in found if not _is_excluded(c)]),
        "not_in_warehouse": len(missing),
    }
    for f, (found, missing) in FAMILIES.items()
])
# MMSC-sourced, so "not_in_warehouse" is meaningless here: these columns are not
# in the usage-trends catalog. §8's join is what validates them.
report.loc[len(report)] = {
    "family": "billing history (MMSC)",
    "requested": len(MMSC_FEATURES),
    "kept": len([c for c in MMSC_FEATURES if not _is_excluded(c)]),
    "not_in_warehouse": 0,
}
report.loc[len(report)] = {
    "family": "TOTAL",
    "requested": report["requested"].sum(),
    "kept": len(FEATURES) + len([c for c in MMSC_FEATURES if not _is_excluded(c)]),
    "not_in_warehouse": report["not_in_warehouse"].sum(),
}
print(report.to_string(index=False))

print("\nnot in the warehouse (generated by convention, absent from the catalog):")
for f, (_found, missing) in FAMILIES.items():
    if missing:
        print(f"  {f}:")
        for c in missing:
            print(f"      {c}")

if _dropped_by_exclusion:
    print(f"\ndropped by the exclusion rules: {sorted(set(_dropped_by_exclusion))}")

# --- PER-FAMILY FLOOR -------------------------------------------------------
# The list above is a report, and a report is not a guard. If the warehouse
# renames a prefix - trailing_3m_ to t3m_, say - every generated name in a family
# lands in `missing`, the family contributes nothing, and the run still succeeds
# with a quietly gutted feature set. Require each family to keep at least half of
# what it asked for. A family legitimately missing one variant (a binary flag
# with no _vs_12m_ago twin) passes; a family that collapsed does not.
MIN_FAMILY_KEEP_SHARE = 0.5

_collapsed = [
    (f, len([c for c in found if not _is_excluded(c)]), len(found) + len(missing))
    for f, (found, missing) in FAMILIES.items()
    if (len(found) + len(missing))
    and len([c for c in found if not _is_excluded(c)])
    < MIN_FAMILY_KEEP_SHARE * (len(found) + len(missing))
]

assert not _collapsed, (
    "feature family/families kept less than "
    f"{MIN_FAMILY_KEEP_SHARE:.0%} of the names generated for them:\n"
    + "\n".join(f"    {f:<32} kept {k} of {r}" for f, k, r in _collapsed)
    + "\nThe naming convention in VARIANT_TEMPLATES no longer matches the "
      "warehouse, or the catalog probe in §5a is incomplete. Reconcile before "
      "extracting - the run would otherwise succeed with a gutted feature set."
)

# --- the trends-sourced set, frozen BEFORE the MMSC features are appended ----
# §7b asserts every one of these exists in USAGE_COLS. The MMSC features come
# from a different table and must be scoped out of that assertion rather than
# silently dropped by it - which is what _flat() would have done to them.
TRENDS_FEATURES = list(FEATURES)

FEATURES = TRENDS_FEATURES + [c for c in MMSC_FEATURES if c not in TRENDS_FEATURES]

print(f"\nFEATURES: {len(FEATURES)} columns "
      f"({len(TRENDS_FEATURES)} from usage-trends + {len(FEATURES) - len(TRENDS_FEATURES)} from MMSC)")
print(f"  billing history (MMSC): {MMSC_FEATURES}")

### 7b. Guardrails

Assertions against the failure modes that would quietly invalidate the model: an excluded column
back in `FEATURES`, a duplicate, or a name the warehouse doesn't have.

The `USAGE_COLS` check is deliberately scoped to `TRENDS_FEATURES`, because the six MMSC columns
come from a different table; two further assertions require `TRENDS_FEATURES` and `MMSC_FEATURES` to
**partition** `FEATURES` exactly, so nothing can hide in the gap.

This cell also defines `KEYS`, `BASELINE` and `PASSTHROUGH` - columns selected by the extract but
kept out of `X_COLS` - and prints the usage-trends columns the spec leaves out, which is the audit
trail for "did we forget a signal?" and the list to revisit for v2.

In [ ]:
_leaks = [c for c in FEATURES if _is_excluded(c)]
assert not _leaks, (
    f"excluded column(s) present in FEATURES: {_leaks}. _is_excluded covers "
    "EXCLUDE_EXACT and EXCLUDE_PATTERNS together, so one of these got in: the "
    "retention health score or a derivative (product_score / "
    "trailing_3m_product_score / delta_product_score_*), a forward-looking "
    "_in_(1|3|6|12)m column, a _date_sfdc column or one derived from it "
    "(months_since_billing_activation / was_ever_billing_customer / "
    "months_since_last_processed_dollars), months_since_last_dm_connect, is_gov, "
    "or the org id. Remove it from §7's family lists - it must not reach FEATURES."
)

_dupes = [c for c in set(FEATURES) if FEATURES.count(c) > 1]
assert not _dupes, f"duplicate columns in FEATURES: {_dupes}"

# SCOPED to the trends-sourced features: the MMSC family lives on a different
# table, so USAGE_COLS is the wrong catalog for it. The two partition assertions
# below close the gap this scoping opens.
_unknown = sorted(set(TRENDS_FEATURES) - USAGE_COLS)
assert not _unknown, f"trends-sourced columns not in the warehouse: {_unknown}"

assert set(TRENDS_FEATURES) | set(MMSC_FEATURES) >= set(FEATURES), (
    "FEATURES contains a column that is neither trends-sourced nor MMSC-sourced: "
    f"{sorted(set(FEATURES) - set(TRENDS_FEATURES) - set(MMSC_FEATURES))}"
)
assert not (set(TRENDS_FEATURES) & set(MMSC_FEATURES)), (
    "a column is claimed by both source tables: "
    f"{sorted(set(TRENDS_FEATURES) & set(MMSC_FEATURES))}"
)
_mmsc_missing = [c for c in MMSC_FEATURES if c not in FEATURES]
assert not _mmsc_missing, (
    f"MMSC features missing from FEATURES: {_mmsc_missing}. They must be appended "
    "after the catalog reconciliation in §7a - _flat() / _expand() drop anything "
    "absent from USAGE_COLS, and these are not in it."
)

print("OK - no excluded columns, no duplicates, every trends-sourced name in the "
      "usage-trends catalog, TRENDS_FEATURES and MMSC_FEATURES partition FEATURES "
      "exactly. Census: §7a above.")

# --- what the spec leaves on the table -------------------------------------
KEYS = ["school_id", "dim_month"]
BASELINE = [c for c in ["product_score"] if c in USAGE_COLS]  # scored against, never trained on

# Selected by the extract alongside the keys but kept out of X_COLS:
# product_score is the baseline, ORG_COL is §12e's diagnostic key. Both are in
# EXCLUDE_EXACT, so neither can reach FEATURES.
PASSTHROUGH = BASELINE + [c for c in [ORG_COL] if c and c in USAGE_COLS]
assert not (set(PASSTHROUGH) & set(FEATURES)), (
    f"passthrough column leaked into FEATURES: {sorted(set(PASSTHROUGH) & set(FEATURES))}"
)
print(f"passthrough (selected, never trained on): {PASSTHROUGH}")

_unused = sorted(USAGE_COLS - set(TRENDS_FEATURES) - set(KEYS) - set(PASSTHROUGH))
print(f"\n{len(_unused)} usage-trends columns not used as features:")
for c in _unused:
    print(f"  {c}{'  [excluded by spec]' if _is_excluded(c) else ''}")

---

## 8. The extract

Built from §6's constants and §7's feature lists - change a constant, re-run, get a new dataset. No
SQL editing. Two cells: the first builds the SQL and checks the parameter binding, the second runs it
into `out`.

Four CTEs and one final select:

- **`obs`** - the population: premium, non-government, in one of the four segments, in US/CA, inside
  the window. Selects `KEYS + TRENDS_FEATURES + PASSTHROUGH`.
- **`events`** - the qualifying cancellation requests, one row per school-month:
  `coalesce(is_reportable_request, false)` **and** a Cancelation Request type **and**
  `is_target_population` **and** `case_created_date is not null`, with
  `event_month = date_trunc('month', case_created_date)::date`.
- **`labels`** - `events` joined forward onto `obs`, aggregated to one row per school-month. A school
  can file more than one request inside a 3-month window, so labelling by a plain join would
  duplicate the observation row once per event.
- **`billing`** - the five MMSC columns for the same window, LEFT JOINed on **both** `school_id` and
  `dim_month`. `has_ever_billed` is derived in SQL from
  `months_since_first_processed_dollars_local is not null`.

A `not exists` against the same `events` CTE drops - rather than labels - any observation row whose
own month contains a qualifying request. The run cell reports what that cost, against a separate
cheap count of the un-dropped population, and reports whether the MMSC join landed.

The three `IN` lists - segments, countries, request types - are passed as `%s` parameters, not
interpolated. Binding is **positional**, so `EXTRACT_PARAMS` has to be in the order the placeholders
appear in the SQL text; the cell asserts both the count and the CTE order, because getting the order
wrong produces a silently wrong filter rather than an error.

In [ ]:
OBS_COLS = KEYS + TRENDS_FEATURES + PASSTHROUGH

_sel_obs = "\n        , ".join(OBS_COLS)
_seg_ph  = ", ".join(["%s"] * len(POPULATION_SEGMENTS))
_ctry_ph = ", ".join(["%s"] * len(POPULATION_COUNTRIES))
_type_ph = ", ".join(["%s"] * len(CANCEL_REQUEST_TYPES))
_excl_reason_ph = ", ".join(["%s"] * len(EXCLUDED_REASON_CATEGORIES))
_target  = "\n      and is_target_population" if APPLY_TARGET_POPULATION else ""

# The obs CTE, shared verbatim by the extract and by the pre-drop count below,
# so the "what did the request-month drop cost" figure is measured against
# exactly the same population the extract started from.
_OBS_WHERE = f"""
    where dim_month >= date '{OBS_START}'
      and dim_month <  date '{OBS_END}'
      and {PREMIUM_FILTER}
      and {GOV_FILTER}
      and customer_segment in ({_seg_ph})
      -- Normalised on both sides: POPULATION_COUNTRIES is already upper-cased
      -- and trimmed, and the column is upper(trim(...))'d to match. §12b prints
      -- the distinct values in this window.
      and upper(trim(school_country)) in ({_ctry_ph})""".strip("\n")

EXTRACT_SQL = f"""
with obs as (
    select
          {_sel_obs}
    from {TABLES['usage_trends']}
{_OBS_WHERE}
)
, events as (
    -- A qualifying CANCELLATION REQUEST: the reportable-request flag, coalesced
    -- because it is nullable on some models; a Cancelation Request type, which
    -- drops most failed payments (the majority of "reportable requests" on
    -- this table); an explicit exclusion of remaining Failed Payment, Business
    -- Changes, and Facility Closed reason categories that still slip through
    -- the type filter (687 rows / 2.83% of positives in the h3 population,
    -- verified against the actual extract population); the target-population
    -- gate; and a non-null case_created_date, the only date this notebook
    -- labels by.
    select distinct
          school_id
        , date_trunc('month', case_created_date)::date as event_month
    from {TABLES['churn_cases']}
    where coalesce({REQUEST_FLAG}, false)
      and summarized_request_type in ({_type_ph}){_target}
      and cancellation_reason_category not in ({_excl_reason_ph})
      and case_created_date is not null
)
, labels as (
    select
          o.school_id
        , o.dim_month
        , 1 as label
    from obs o
    join events e
      on e.school_id   = o.school_id
     and e.event_month >  o.dim_month
     and e.event_month <= dateadd(month, {LABEL_HORIZON_MONTHS}, o.dim_month)
    group by 1, 2
)
, billing as (
    -- MMSC, same school_id x dim_month grain as obs. Every column here is
    -- documented "as of dim_month".
    select
          school_id
        , dim_month
        , months_since_first_processed_dollars_local
        , processed_dollars_rank_local
        , has_processed_dollars_this_month_local
        , has_processed_dollars_last_month_local
        , monthly_billing_status_local
    from {TABLES['billing_history']}
    where dim_month >= date '{OBS_START}'
      and dim_month <  date '{OBS_END}'
)
select
      o.*
    , b.months_since_first_processed_dollars_local
    , b.processed_dollars_rank_local
    , b.has_processed_dollars_this_month_local
    , b.has_processed_dollars_last_month_local
    , b.monthly_billing_status_local
    , case
        when b.months_since_first_processed_dollars_local is not null then 1
        else 0
      end as has_ever_billed
    , coalesce(l.label, 0) as label
from obs o
left join labels l
       on l.school_id = o.school_id
      and l.dim_month = o.dim_month
left join billing b
       on b.school_id = o.school_id
      and b.dim_month = o.dim_month
-- Drop the request month itself rather than labelling it 0. Anti-join against
-- the SAME events CTE the label is built from, so the two can never disagree
-- about what a qualifying request is.
where not exists (
    select 1
    from events e2
    where e2.school_id   = o.school_id
      and e2.event_month = o.dim_month
)
"""

# Positional: %s placeholders are filled left-to-right in the order they appear
# in the SQL text. The obs CTE comes first (segments, then countries), the events
# CTE second (request types, then excluded reason categories). Reorder this
# tuple and you get a filter that is wrong but does not error.
EXTRACT_PARAMS = (
    tuple(POPULATION_SEGMENTS)
    + tuple(POPULATION_COUNTRIES)
    + tuple(CANCEL_REQUEST_TYPES)
    + tuple(EXCLUDED_REASON_CATEGORIES)
)

_n_ph = EXTRACT_SQL.count("%s")
assert _n_ph == len(EXTRACT_PARAMS), (
    f"{_n_ph} %s placeholders in the SQL but {len(EXTRACT_PARAMS)} params - "
    "the IN lists and EXTRACT_PARAMS have drifted apart"
)

_ph_order = [
    ("customer_segment",              EXTRACT_SQL.index("customer_segment in (")),
    ("school_country",                EXTRACT_SQL.index("upper(trim(school_country)) in (")),
    ("summarized_request_type",       EXTRACT_SQL.index("summarized_request_type in (")),
    ("cancellation_reason_category",  EXTRACT_SQL.index("cancellation_reason_category not in (")),
]
assert [p for p, _ in _ph_order] == [p for p, _ in sorted(_ph_order, key=lambda kv: kv[1])], (
    f"IN lists appear in the SQL in a different order than EXTRACT_PARAMS builds "
    f"them: {_ph_order}"
)
print(f"placeholders {_n_ph} == params {len(EXTRACT_PARAMS)}  OK")
print("param order  segments -> countries -> request types -> excluded reasons  OK\n")

OBS_COUNT_SQL = f"""
with obs as (
    select
          school_id
        , dim_month
    from {TABLES['usage_trends']}
{_OBS_WHERE}
)
select count(*) as n from obs
"""
OBS_COUNT_PARAMS = tuple(POPULATION_SEGMENTS) + tuple(POPULATION_COUNTRIES)
assert OBS_COUNT_SQL.count("%s") == len(OBS_COUNT_PARAMS)

_lines = EXTRACT_SQL.strip().splitlines()
_from = next(i for i, l in enumerate(_lines) if l.startswith("    from "))
print("\n".join(_lines[:4]))
print(f"        ... {_from - 4} more columns ...")
print("\n".join(_lines[_from:]))

In [ ]:
# `out` is the ONE frame for the rest of the notebook. Everything downstream
# mutates it in place instead of taking a .copy() - at this row count a copy per
# stage is hundreds of MB of RSS. Every later cell is written to be idempotent
# (re-assigning a column with the same value is a no-op), so you can re-run any
# of them without re-running this extract. Re-run THIS cell for a clean frame.
_t0 = time.time()
out = run_query(EXTRACT_SQL, params=EXTRACT_PARAMS)
print(f"{len(out):,} rows x {out.shape[1]} cols in {time.time() - _t0:.1f}s")

# Every IN-list filter here can match nothing without raising, and the notebook
# downstream would then run cleanly on zero rows and report a 0% base rate.
assert len(out) > 0, (
    "The extract returned ZERO rows. This is almost certainly a filter that "
    "matched nothing rather than a genuinely empty population. In order of "
    "likelihood:\n"
    f"  1. POPULATION_COUNTRIES = {POPULATION_COUNTRIES} does not match the "
    "stored school_country format - the predicate is "
    "upper(trim(school_country)) in (...). RUN THE APPENDIX COUNTRY CHECK, "
    "§12b: it lists the actual distinct values.\n"
    f"  2. POPULATION_SEGMENTS = {POPULATION_SEGMENTS} does not match the "
    "stored customer_segment strings - §12d prints the real ones.\n"
    f"  3. The window {OBS_START} -> {OBS_END} has no premium rows, or "
    f"{PREMIUM_FILTER!r} / {GOV_FILTER!r} is not a valid boolean predicate on "
    "this table."
)

# --- permanently-closing schools -------------------------------------------
# UNSAVEABLE, not unpredictable. A school-month carrying a permanently-closing
# stoppage reason requests at 27% against a 3.1% base - an 8.8x lift the model
# will happily chase - and 76% of those months had never filed a request before,
# so the signal genuinely leads the request rather than echoing it. But the
# school is closing. A rep handed one has spent a call they could never win, and
# precision@k looks great while it happens. Same logic as excluding the micro and
# Enterprise segments: not a saveable event, so not in the population.
#
#   "from_flag"     drop a school's months from the first closing month ONWARD.
#                   Replicable at scoring time from what you know then, so the
#                   evaluation stays honest. Costs ~0.7% of rows, ~2% of positives.
#   "whole_school"  drop every month of any school that ever closes. Cleaner
#                   training signal, but it uses future information - a school
#                   flagged in Dec 2025 also loses its Jan 2024 rows, which you
#                   could not have known to skip. ~1.9% of rows, ~4.3% of positives.
#   None            keep them, and read precision@k knowing what is in it.
EXCLUDE_PERMANENTLY_CLOSING = "from_flag"
_CLOSING_COL = "reason_school_permanently_closing_count"

# Row count as extracted, before this exclusion - the request-month diagnostic
# below diffs against the SQL population, which knows nothing about this filter.
_rows_from_sql = len(out)

# fetch_dataframe hands back datetime.date objects; the comparison below needs
# real datetimes. §9 does this too and it is idempotent.
out["dim_month"] = pd.to_datetime(out["dim_month"])

if EXCLUDE_PERMANENTLY_CLOSING and _CLOSING_COL in out.columns:
    _before_rows, _before_pos = len(out), int(out["label"].sum())
    _c = out[_CLOSING_COL].fillna(0) > 0

    if EXCLUDE_PERMANENTLY_CLOSING == "whole_school":
        _drop_schools = set(out.loc[_c, "school_id"].unique())
        _keep = ~out["school_id"].isin(_drop_schools)
    elif EXCLUDE_PERMANENTLY_CLOSING == "from_flag":
        # First closing month per school, then drop that month and everything
        # after it. Schools that never close have a NaT threshold and are kept.
        _first = out.loc[_c].groupby("school_id")["dim_month"].min()
        _thresh = out["school_id"].map(_first)
        _keep = ~(_thresh.notna() & (out["dim_month"] >= _thresh))
    else:
        raise ValueError(
            f"EXCLUDE_PERMANENTLY_CLOSING={EXCLUDE_PERMANENTLY_CLOSING!r} - expected "
            "'from_flag', 'whole_school' or None"
        )

    out = out[_keep].reset_index(drop=True)
    _dr, _dp = _before_rows - len(out), _before_pos - int(out["label"].sum())
    print(f"\npermanently-closing exclusion ({EXCLUDE_PERMANENTLY_CLOSING}): "
          f"dropped {_dr:,} rows ({_dr / max(_before_rows, 1):.1%}) and "
          f"{_dp:,} positives ({_dp / max(_before_pos, 1):.1%})")
    print(f"  {len(out):,} rows remain, {int(out['label'].sum()):,} positives, "
          f"rate {out['label'].mean():.2%}")
    assert len(out) > 0, "the permanently-closing exclusion removed every row"
elif EXCLUDE_PERMANENTLY_CLOSING:
    print(f"\nWARNING: EXCLUDE_PERMANENTLY_CLOSING is set but {_CLOSING_COL} is not "
          "in the frame - closing schools are still in the population.")

# --- what the request-month drop cost --------------------------------------
# Measured against the pre-exclusion population, since the count query below
# does not know about the closing filter applied just above.
# The anti-join is invisible in `out` by construction, so count the population
# without it and diff.
_obs_rows = int(run_query(OBS_COUNT_SQL, params=OBS_COUNT_PARAMS).iloc[0, 0])
_dropped_request_months = _obs_rows - _rows_from_sql
print(f"\nrequest-month rows dropped: {_dropped_request_months:,} of {_obs_rows:,} "
      f"observation rows ({_dropped_request_months / max(_obs_rows, 1):.3%})")
if _dropped_request_months < 0:
    print("  WARNING: negative - the count query and the extract disagree about the")
    print("  population. Did a constant change between running the two cells?")

# --- did the MMSC join land? ------------------------------------------------
# A fan-out would inflate the row count (caught by §9's grain assertion); a join
# that matched nothing is silent and looks exactly like a population that never
# billed. Distinguish the two:
#   no MMSC row at all   -> BOTH months_since_first_processed_dollars_local and
#                           monthly_billing_status_local are null (failed join)
#   MMSC row, no billing -> monthly_billing_status_local = '6. Never Billing'
_no_mmsc_row = int(
    out[["months_since_first_processed_dollars_local",
         "monthly_billing_status_local"]].isna().all(axis=1).sum()
)
_never_billing = int((out["monthly_billing_status_local"] == "6. Never Billing").sum())
print(f"\nMMSC join: {_no_mmsc_row:,} rows with NO MMSC row at all "
      f"({_no_mmsc_row / len(out):.2%})  <- failed join, not a fact about billing")
print(f"           {_never_billing:,} rows with an MMSC row saying "
      f"'6. Never Billing' ({_never_billing / len(out):.2%})")
print(f"           has_ever_billed = 1 on {int(out['has_ever_billed'].sum()):,} rows "
      f"({out['has_ever_billed'].mean():.2%})")
if _no_mmsc_row > 0.10 * len(out):
    print("  WARNING: >10% of observation rows got no MMSC row. Check that MMSC")
    print("  really is school_id x dim_month over this window before trusting")
    print("  has_ever_billed - an unmatched row is NaN, not 'never billed'.")

out.head()

---

## 9. The 2x2 school-by-time split

Two **independent** partitions, crossed, written onto `out` as `school_group` and `split`.

1. **Schools** - every distinct `school_id` is assigned once, disjointly, to group `"A"`
   (`TRAIN_SCHOOL_FRACTION`) or group `"B"`, seeded on `SCHOOL_SPLIT_SEED` and **stratified** on
   whether that school has any `label == 1` row anywhere in the extract. Assigned in pandas rather
   than SQL, so it is reproducible, inspectable, and every month of a given school lands in the same
   group.
2. **Time** - `TRAIN_YEAR`, `TEST_YEAR`, and everything after that.

| `split` | rule | role |
| --- | --- | --- |
| `train` | group A, year == 2024 | the only cell fitted on |
| `holdout_school` | group B, year == 2024 | new schools, same era |
| `holdout_time` | group A, year == 2025, month >= `EVAL_START` | same schools, new era |
| `test` | group B, year == 2025, month >= `EVAL_START` | new schools, new era - the ship gate |
| `embargo` | year == 2025, month < `EVAL_START` | excluded; labelled rather than deleted, so the cost is visible |
| `retrain_only` | year > 2025 | answerable 2026 months for the live retrain; never evaluated |

`EVAL_START` is derived in §6. The cell asserts that the six cells partition the frame exactly, that
`train` is school-disjoint from `test` and from `holdout_school`, and that `holdout_time` and `test`
span **identical months**.

The per-cell table it prints - rows, schools, positives, positive rate and month span for all six
cells - is the **canonical** split composition. §10 and §11 refer here rather than reprinting it.

In [ ]:
# Mutates `out` in place - no copy. Every assignment below is deterministic
# (seeded) and idempotent, so this cell is safe to re-run without re-running the
# extract, and re-running it gives the identical split.
out["dim_month"] = pd.to_datetime(out["dim_month"])

# --- grain -----------------------------------------------------------------
# Also the fan-out guard for §8's MMSC join: MMSC is documented at
# school_id x dim_month, but if it ever carried two rows for one school-month the
# LEFT JOIN would silently duplicate observation rows, and this is where that
# shows up.
_dupes = out.duplicated(subset=KEYS).sum()
assert _dupes == 0, (
    f"{_dupes:,} duplicate school-month rows - either the label join or the MMSC "
    "billing join fanned out. Check that both are one row per school x month."
)
print(f"grain OK: one row per school-month ({len(out):,} rows, "
      f"{out['school_id'].nunique():,} schools)")

# --- 1. the school partition: disjoint A / B, stratified on ever-positive ---
# Assigned in pandas, not SQL: reproducible from the seed, inspectable, and
# guaranteed to put every month of a school on the same side.
_ever_pos = out.groupby("school_id")["label"].max().astype(int)
_rng = np.random.default_rng(SCHOOL_SPLIT_SEED)

_group_of = {}
_strata_report = []
for _flag, _stratum in ((1, "ever positive"), (0, "never positive")):
    _ids = sorted(_ever_pos.index[_ever_pos == _flag].tolist())
    _perm = _rng.permutation(len(_ids))
    _n_a = int(round(TRAIN_SCHOOL_FRACTION * len(_ids)))
    _a_ids = {_ids[i] for i in _perm[:_n_a]}
    for _sid in _ids:
        _group_of[_sid] = "A" if _sid in _a_ids else "B"
    _strata_report.append((_stratum, len(_ids), _n_a, len(_ids) - _n_a))

out["school_group"] = out["school_id"].map(_group_of)
assert out["school_group"].isin(("A", "B")).all(), "a school got no A/B group"

_sg = out.drop_duplicates("school_id")[["school_id", "school_group"]]
_sg_pos = _sg["school_id"].map(_ever_pos)

print(f"\nschool partition (seed {SCHOOL_SPLIT_SEED}, "
      f"target A share {TRAIN_SCHOOL_FRACTION:.0%}, stratified on ever-positive):")
print(f"  {'stratum':<16}{'schools':>9}{'-> A':>9}{'-> B':>9}")
for _stratum, _n, _na, _nb in _strata_report:
    print(f"  {_stratum:<16}{_n:>9,}{_na:>9,}{_nb:>9,}")
print(f"  {'TOTAL':<16}{len(_sg):>9,}"
      f"{int((_sg['school_group'] == 'A').sum()):>9,}"
      f"{int((_sg['school_group'] == 'B').sum()):>9,}")
print("  ever-positive share   "
      + "   ".join(f"{_g} {_sg_pos[_sg['school_group'] == _g].mean():.2%}"
                   for _g in ("A", "B"))
      + "   <- stratification check (should be close)")

# --- 2. the time partition, crossed with the school partition --------------
# Masks are written out per cell exactly as the table in the markdown above, so
# that mutual exclusivity and exhaustiveness can be ASSERTED rather than trusted
# to the order of an if/elif chain.
_grp   = out["school_group"]
_year  = out["dim_month"].dt.year
_month = out["dim_month"]

SPLIT_MASKS = {
    "train":          (_grp == "A") & (_year == TRAIN_YEAR),
    "holdout_school": (_grp == "B") & (_year == TRAIN_YEAR),
    "holdout_time":   (_grp == "A") & (_year == TEST_YEAR) & (_month >= EVAL_START),
    "test":           (_grp == "B") & (_year == TEST_YEAR) & (_month >= EVAL_START),
    "embargo":        (_year == TEST_YEAR) & (_month < EVAL_START),
    "retrain_only":   (_year > TEST_YEAR),
}
SPLITS = list(SPLIT_MASKS)

_hits = sum(m.astype("int8") for m in SPLIT_MASKS.values())
assert (_hits == 1).all(), (
    f"{int((_hits > 1).sum()):,} rows match more than one split cell and "
    f"{int((_hits == 0).sum()):,} match none. The six cells must partition the "
    f"frame exactly. Unassigned months: "
    f"{sorted(out.loc[_hits == 0, 'dim_month'].dt.strftime('%Y-%m').unique())} "
    f"(anything before {TRAIN_YEAR} would land here - OBS_START is {OBS_START})"
)

out["split"] = np.select(list(SPLIT_MASKS.values()), SPLITS, default="UNASSIGNED")
assert not (out["split"] == "UNASSIGNED").any(), "np.select left rows unassigned"

# --- school-disjointness ----------------------------------------------------
_ids_in = {s: set(out.loc[out["split"] == s, "school_id"]) for s in SPLITS}
for _a, _b in (("train", "test"), ("train", "holdout_school")):
    _shared = _ids_in[_a] & _ids_in[_b]
    assert not _shared, (
        f"{len(_shared):,} school_ids appear in BOTH '{_a}' and '{_b}' - the A/B "
        f"partition is not disjoint. Example: {sorted(_shared)[:5]}"
    )
print(f"\nschool-disjoint: train n test = 0, train n holdout_school = 0  OK")

# --- the two 2025 cells must cover the SAME months --------------------------
# Otherwise holdout_time vs test is not a clean school-effect comparison.
_span = {s: (out.loc[out["split"] == s, "dim_month"].min(),
             out.loc[out["split"] == s, "dim_month"].max())
         for s in SPLITS if (out["split"] == s).any()}
_ht, _te = _span.get("holdout_time"), _span.get("test")
assert _ht == _te, (
    f"holdout_time spans {_ht} but test spans {_te} - the two 2025 cells must "
    "cover identical months or the school effect and the time effect stay confounded"
)
if _ht:
    _ht_months = set(out.loc[out["split"] == "holdout_time", "dim_month"].unique())
    _te_months = set(out.loc[out["split"] == "test", "dim_month"].unique())
    assert _ht_months == _te_months, "holdout_time / test month SETS differ"
    print(f"\nholdout_time and test both span {_ht[0]:%Y-%m} -> {_ht[1]:%Y-%m} "
          f"({len(_ht_months)} months, identical)  OK")

# --- per-cell report -------------------------------------------------------
print(f"\nembargo: {TEST_YEAR} months before {EVAL_START:%Y-%m} "
      f"(derived from a {LABEL_HORIZON_MONTHS}-month horizon)\n")
_summary = (out.assign(_s=pd.Categorical(out["split"], categories=SPLITS, ordered=True))
               .groupby("_s", observed=False)
               .agg(rows=("label", "size"),
                    schools=("school_id", "nunique"),
                    positives=("label", "sum"),
                    pos_rate=("label", "mean"),
                    first_month=("dim_month", "min"),
                    last_month=("dim_month", "max")))
_summary["pos_rate"] = (_summary["pos_rate"] * 100).round(2).astype(str) + "%"
_summary["first_month"] = _summary["first_month"].dt.strftime("%Y-%m")
_summary["last_month"] = _summary["last_month"].dt.strftime("%Y-%m")
_summary.index.name = "split"
print(_summary.to_string())

_tr = out[out["split"] == "train"]

# --- imbalance -------------------------------------------------------------
_pos = int(_tr["label"].sum())
_neg = len(_tr) - _pos
print(f"\ntrain positives {_pos:,} / negatives {_neg:,}")
print(f"scale_pos_weight starting point (neg/pos): {_neg / max(_pos, 1):.1f}")

# Null accounting is in §10, next to the coercion that can invent nulls.

---

## 10. Save

Names `X_COLS` (the candidate set handed to feature selection) and `Y_COL`, conditions the numeric
columns, and writes the canonical readable parquet to `RAW_DIR`. `PASSTHROUGH` rides along in the
frame but is deliberately outside `X_COLS`.

Numeric conditioning, in this fixed order:

1. **Coerce** every model-numeric column with `pd.to_numeric(errors="coerce")`, and print a WARNING
   block naming any column where the coercion *introduced* nulls, with counts. Which columns count
   as numeric is decided by **exclusion** - everything except `KEYS`, `CATEGORICAL`, the label, the
   identifier passthroughs and §11's dummies - because `fetch_dataframe` builds the frame from raw
   Python objects, so a Redshift `DECIMAL` can land in an object column that a dtype-based rule would
   skip entirely.
2. **`+inf` / `-inf` -> NaN**, with a per-column count.
3. **Clip to the float32 representable range** (`+/-np.finfo(np.float32).max`), with a per-column
   count of what was clipped and at which end.
4. **Round to `DECIMALS` places, in float64** - before the float32 cast.
5. **Cast to float32**, and the label to `int8`.
6. **Assert no float column contains an infinity.** A tripwire on step 3, not on the data; NaN is
   expected and tolerated.

Nulls are preserved as NaN throughout - no fill, no sentinel, categoricals included - and the cell
prints the surviving null count per column plus how many `delta_*` cells hold exactly `-1.0`. The
parquet round trip is asserted to keep the same total null count.

Then a dtype gate before the write: the categoricals are cast to `category` with **`str` categories**
and their nulls left as NaN, the frame is asserted to contain **no object column** (naming any
offender and the Python type of its first non-null value), and every remaining dtype is asserted to
be one of `float32`, `int8`, `int64`, `datetime64[ns]`, `category`, `bool`. A dtype census prints
either way.

In [ ]:
X_COLS = FEATURES     # candidates. PASSTHROUGH (product_score, org id) is excluded by construction.
Y_COL  = "label"

DECIMALS = 5     # float precision kept in the parquet

# The largest finite float32. Anything beyond it overflows to +/-inf during the
# astype("float32") below - silently, with a RuntimeWarning rather than an
# exception - so out-of-range values are clipped to this instead.
FLOAT32_MAX = float(np.finfo(np.float32).max)   # ~3.4028235e38

# Mutates `out` in place - no copy. Re-runnable: with no fill step, the only
# state-dependent diagnostics are the coercion / infinity / clip counters, and
# each correctly reports zero on a second run.
CATEGORICAL = (
    [c for c in DESCRIPTIVE if c in out.columns]
    + [c for c in MMSC_CATEGORICAL if c in out.columns]
    + ["split", "school_group"]
)

# Identifier passthroughs: strings (possibly UUIDs), never numeric, never
# features. They get the same str-safe category cast as CATEGORICAL just before
# the write - see the dtype gate at the bottom of this cell.
ID_PASSTHROUGH = [c for c in ["school_id"] + ([ORG_COL] if ORG_COL else [])
                  if c in out.columns]

# dim_month is converted to datetime64 in §9, which runs first, and the
# conversion is idempotent - so it is NOT repeated here. If §9 were somehow
# skipped, dim_month would still be a column of datetime.date objects, and the
# object-dtype gate at the bottom of this cell rejects it by name.

# --- 0. which columns are model numerics, by EXCLUSION ---------------------
# NOT by dtype: fetch_dataframe infers dtypes from raw Python objects, so a
# DECIMAL arrives as decimal.Decimal in an OBJECT column that
# select_dtypes(include=["floating"]) does not match. Excluded, and why:
#   KEYS            school_id (a string id) and dim_month (a datetime)
#   CATEGORICAL     the descriptive strings + split + school_group
#   Y_COL           the label, cast to int8 on its own below
#   ID_PASSTHROUGH  school_id / the org id - may be a UUID, must NOT be coerced
# product_score is deliberately NOT excluded: it is a numeric passthrough
# baseline and belongs in the numeric set.
# If §11 has already run in this kernel its one-hot dummies are sitting on `out`,
# named <categorical>__<...>. They are uint8 indicators, not measurements:
# coercing them would cast them to float32 and break the dtype contract §11
# asserts. Excluded here, and excluded again from the written frame at the bottom
# of this cell, so re-running §10 alone after §11 is harmless.
_DUMMY_LIKE = [c for c in out.columns
               if any(c.startswith(f"{k}__") for k in CATEGORICAL)]

_NON_NUMERIC = (set(KEYS) | set(CATEGORICAL) | {Y_COL}
                | set(ID_PASSTHROUGH) | set(_DUMMY_LIKE))
_float_cols = [c for c in out.columns if c not in _NON_NUMERIC]
_delta_cols = [c for c in _float_cols if c.startswith("delta_")]
print(f"numeric columns by exclusion: {len(_float_cols)} of {out.shape[1]} "
      f"({len(_NON_NUMERIC & set(out.columns))} excluded: keys, categoricals, "
      "label, org id)")

# --- 1. coerce to numeric, LOUDLY ------------------------------------------
# pd.to_numeric(errors="coerce") turns anything non-numeric into NaN without
# complaint, so count nulls per column before and after and report every column
# where the coercion INVENTED a null.
_coerce_report = []
for c in _float_cols:
    _before = int(out[c].isna().sum())
    out[c] = pd.to_numeric(out[c], errors="coerce")
    _after = int(out[c].isna().sum())
    if _after > _before:
        _coerce_report.append((c, _after - _before))

_coerce_total = sum(n for _, n in _coerce_report)
print(f"\ncoerced {len(_float_cols)} columns with pd.to_numeric(errors='coerce'): "
      f"{_coerce_total:,} value(s) were not numeric")
if _coerce_report:
    print("!" * 72)
    print("WARNING: coercion INTRODUCED nulls - these values were not numeric and")
    print("errors='coerce' turned them into NaN:")
    for c, n in sorted(_coerce_report, key=lambda kv: -kv[1]):
        print(f"    {c:<52} {n:>10,}  ({n / len(out):.4%} of rows)")
    print(f"  {_coerce_total:,} value(s) across {len(_coerce_report)} column(s).")
    print("  Check these columns in the warehouse before trusting the dataset.")
    print("!" * 72)
else:
    print("  no column gained a null: every value in every numeric column parsed")

# --- 2. infinities -> NaN --------------------------------------------------
_inf_counts = {
    c: int(np.isinf(out[c].to_numpy(dtype="float64", na_value=np.nan)).sum())
    for c in _float_cols
}
_inf_counts = {c: n for c, n in _inf_counts.items() if n}
_inf_total = sum(_inf_counts.values())
print(f"\nnon-finite (+/-inf) values found: {_inf_total:,} "
      f"across {len(_inf_counts)} of {len(_float_cols)} numeric columns")
for c, n in sorted(_inf_counts.items(), key=lambda kv: -kv[1]):
    print(f"    {c:<52} {n:>10,}  ({n / len(out):.4%} of rows)")
if _inf_total:
    out[_float_cols] = out[_float_cols].replace([np.inf, -np.inf], np.nan)
    print(f"  -> {_inf_total:,} replaced with NaN")

# --- 3. clip to the float32 representable range ----------------------------
# A finite float64 larger than FLOAT32_MAX overflows to inf in the cast at step
# 5. NaN passes through untouched - np.clip propagates it.
_clip_report = []
for c in _float_cols:
    _v = out[c].to_numpy(dtype="float64", na_value=np.nan)
    _hi = int(np.count_nonzero(_v > FLOAT32_MAX))
    _lo = int(np.count_nonzero(_v < -FLOAT32_MAX))
    if _hi or _lo:
        _clip_report.append((c, _lo, _hi))
        out[c] = np.clip(_v, -FLOAT32_MAX, FLOAT32_MAX)

_clip_total = sum(lo + hi for _, lo, hi in _clip_report)
print(f"\nout-of-float32-range values clipped to +/-{FLOAT32_MAX:.7e}: "
      f"{_clip_total:,} across {len(_clip_report)} of {len(_float_cols)} numeric columns")
for c, _lo, _hi in sorted(_clip_report, key=lambda t: -(t[1] + t[2])):
    print(f"    {c:<52} {_lo:>8,} at the -ve bound, {_hi:>8,} at the +ve bound")

# --- nulls are KEPT ---------------------------------------------------------
# One isna() pass over the frame, reused for the total, the column count and the
# per-column profile. This is THE null summary for the notebook; the only other
# null accounting is the coercion before/after diff above and the parquet
# round-trip check below.
_isna = out.isna()
_nulls_per_col = _isna.sum().loc[lambda s: s > 0].sort_values(ascending=False)
_nulls_kept = int(_isna.to_numpy().sum())
_real_minus_one = int((out[_delta_cols] == -1.0).to_numpy().sum()) if _delta_cols else 0
print(f"\nnulls preserved as NaN: {_nulls_kept:,} cells across "
      f"{len(_nulls_per_col)} of {out.shape[1]} columns (no fill, no sentinel)")
print(f"cells holding exactly -1.0 in the {len(_delta_cols)} delta_* columns: "
      f"{_real_minus_one:,}")
if len(_nulls_per_col):
    print("\n  null rate by column (top 40):")
    for _c, _s in ((_nulls_per_col / len(out)).mul(100).round(1)
                   .head(40).items()):
        print(f"    {_c:<52}{_s:>7.1f}%")

# --- 4. round, then 5. cast ------------------------------------------------
# Rounded in float64 ON PURPOSE. np.round(x, 5) is implemented as
# rint(x * 1e5) / 1e5, and in float32 that multiply OVERFLOWS to inf for any
# value near the float32 max - i.e. for exactly the values step 3 just clipped.
# On a second run of this cell the columns are already float32, so rounding in
# place would resurrect the infinity the clip removed and trip the tripwire
# below. float64 has the headroom (3.4e38 * 1e5 = 3.4e43, nowhere near 1.8e308),
# and the cast back to float32 returns the clipped bound unchanged.
out[_float_cols] = out[_float_cols].astype("float64").round(DECIMALS)
for c in _float_cols:
    out[c] = out[c].astype("float32")
out[Y_COL] = out[Y_COL].astype("int8")

# --- 6. the tripwire -------------------------------------------------------
# Asserts against INFINITY specifically and tolerates NaN -
# np.isfinite(...).all() is False for NaN, so it would fail on every
# legitimately-null column now that nulls are preserved.
for c in _float_cols:
    assert not np.isinf(out[c].to_numpy()).any(), (
        f"{c} contains an infinity after astype('float32'). The clip to "
        f"+/-{FLOAT32_MAX:.7e} in step 3 should have made this impossible, so the "
        "CLIP LOGIC IS BROKEN, not the data: check that step 3 covers this column "
        "and that the clipped values were assigned back to the frame. (NaN is "
        "expected here and is not what this assertion is about.)"
    )
print(f"\nrounded to {DECIMALS} dp, cast {len(_float_cols)} columns to float32, "
      "no infinities (NaN preserved).")

# --- 7. Arrow-native dtypes, guaranteed then asserted ----------------------


def _str_category(s: pd.Series) -> pd.Series:
    """Cast to category with str categories, leaving nulls as NaN.

    NOT s.astype(str): that renders NaN as the literal string "nan", which then
    becomes a real category and corrupts a column whose missingness is
    meaningful. Cast only where the value is non-null.
    """
    if isinstance(s.dtype, pd.CategoricalDtype):
        s = s.astype(object)
    return s.where(s.isna(), s.astype(str)).astype("category")


for c in CATEGORICAL:
    out[c] = _str_category(out[c])

# Identifiers are handled CONDITIONALLY, unlike the categoricals above. If the
# warehouse hands back an integer key, int64 is already Arrow-native and is the
# better dtype downstream - a category merges badly against an int key and
# groupby on it yields every unused category. Only rescue an ID that arrived as
# an object (a UUID or a string key), which is the case the dtype gate below
# would otherwise reject.
for c in ID_PASSTHROUGH:
    if out[c].dtype == object:
        out[c] = _str_category(out[c])
        print(f"  {c}: object -> category (string key)")

_obj_cols = [c for c in out.columns if out[c].dtype == object]
# Built over the offenders only (normally none), so the failure message names
# the column AND the Python type behind it.
_detail = [
    f"    {c:<52} first non-null value is type "
    f"{type(out[c].dropna().iloc[0]).__name__ if out[c].notna().any() else 'ALL-NULL'}"
    for c in _obj_cols
]
assert not _obj_cols, (
    "object-dtype column(s) still present immediately before to_parquet - "
    "pyarrow will fail with a message naming a Python type rather than a column. "
    "Offenders and the type of their first non-null value:\n" + "\n".join(_detail)
)

# Everything must be a dtype Arrow maps cleanly, and there are only six.
ALLOWED_DTYPES = {"float32", "int8", "int64", "datetime64[ns]", "category", "bool"}
_census = out.dtypes.astype(str).value_counts()
print("\ndtype census:")
for _dt, _n in _census.items():
    print(f"    {_dt:<20} {_n:>4} column(s){'' if _dt in ALLOWED_DTYPES else '   <- NOT ALLOWED'}")

_bad_dtypes = [(c, str(out[c].dtype)) for c in out.columns
               if str(out[c].dtype) not in ALLOWED_DTYPES]
assert not _bad_dtypes, (
    "column(s) with a dtype outside the parquet allowlist "
    f"{sorted(ALLOWED_DTYPES)}:\n"
    + "\n".join(f"    {c:<52} {d}" for c, d in _bad_dtypes)
)
print(f"dtypes OK: {out.shape[1]} columns, all within {sorted(ALLOWED_DTYPES)}")

OUT = RAW_DIR / (
    f"churn_v1_{REQUEST_FLAG}_{OBS_START[:7].replace('-', '')}"
    f"_{LAST_ANSWERABLE_MONTH:%Y%m}_h{LABEL_HORIZON_MONTHS}.parquet"
)
# This file is the READABLE dataset: raw categoricals, no model matrix. §11
# writes the encoded twin separately. If §11 already ran, drop its dummies here
# rather than shipping a "readable" parquet that silently contains the matrix.
_readable_cols = [c for c in out.columns if c not in set(_DUMMY_LIKE)]
if _DUMMY_LIKE:
    print(f"\nexcluded {len(_DUMMY_LIKE)} one-hot column(s) left on `out` by §11 - "
          "this parquet stays the readable one")

out[_readable_cols].to_parquet(OUT, index=False)

# Counted over the columns actually written, so the check still holds if §11's
# dummies were dropped above.
_nulls_written = int(out[_readable_cols].isna().to_numpy().sum())
_back = pd.read_parquet(OUT)
_nulls_after = int(_back.isna().to_numpy().sum())
assert _nulls_after == _nulls_written, (
    f"{_nulls_written:,} nulls before the save, {_nulls_after:,} after the round trip - "
    "something is filling or dropping nulls on the way to parquet"
)
print(f"round trip OK: {_nulls_after:,} nulls survive the parquet")

print(f"saved      {OUT.name}")
print(f"on disk    {OUT.stat().st_size / 1e6:.1f} MB")
print(f"in memory  {out.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"\nrows       {len(out):,}")
print(f"X_COLS     {len(X_COLS)} candidate features")
print(f"Y_COL      {Y_COL}")
print(f"BASELINE   {BASELINE}")
print(f"PASSTHRU   {PASSTHROUGH} (never in X_COLS)")
print(f"splits     {len(SPLITS)} cells - composition is §9's per-cell table")
print("\nNext: §11 encodes the categorical features into the model matrix and writes")
print("MODEL_COLS plus a JSON encoding schema.")

---

## 11. Encode the categoricals for the model matrix

§10's parquet is the canonical readable dataset and does not change - the categorical features stay
`category` dtype there, so `customer_segment` reads as `'10-39 small'` in an audit. This section adds
one-hot columns to the same `out` frame, names the authoritative feature list `MODEL_COLS`, and
writes two more files.

- **What gets encoded** is derived as every column that is in `X_COLS` **and** has `category` dtype
  - not from §10's `CATEGORICAL`, which also contains `split` and `school_group`. Expect exactly
  four: `customer_segment`, `monthly_billing_status`, `school_country`,
  `monthly_billing_status_local`. A forbidden-set assertion names any bookkeeping column, key,
  passthrough or label that reached the list anyway.
- **Levels come from the `train` cell only.** `CAT_LEVELS[c]` is the sorted distinct non-null values
  of `c` inside `split == "train"`. Sorted for determinism, because the column order is part of the
  contract a scoring job replays.
- **Three mutually exclusive groups per column:** `c__<slug(level)>` for each train level, `c__other`
  for a non-null value that is not a train level, and `c__isna` for a null. `__other` and `__isna`
  are emitted **always**, even when all-zero on this extract, so a one-row scoring batch produces the
  same columns as the training matrix. Every row's indicators for a column sum to exactly 1, checked
  by `assert_one_hot_invariant`, which a scoring job can reuse on the matrix it builds.
- **Slugging:** lowercase, every run of non-alphanumeric characters becomes a single `_`, leading and
  trailing `_` stripped - `'10-39 small'` -> `10_39_small`, `'1. New'` -> `1_new`. It is lossy, so a
  collision between two levels of one column is asserted rather than silently merged, as is a
  generated name colliding with a column already in `out`.
- **No scaling** is applied. The only transform this notebook persists is the encoding itself.
- **`MODEL_COLS`** is the non-categorical members of `X_COLS` in their existing order, then the
  dummies grouped by source column (levels in `CAT_LEVELS` order, then `__other`, then `__isna`).
  The raw categorical columns stay in `out` and in the encoded file for readability; they are simply
  not in `MODEL_COLS`.

| file | contents | audience |
| --- | --- | --- |
| `..._h3.parquet` (§10) | readable, categoricals as `category` | audits, rep-facing output - **canonical** |
| `..._h3_encoded.parquet` | `KEYS` + `label` + `split` + `school_group` + passthroughs + raw categoricals + all of `MODEL_COLS` | model fitting |
| `..._h3_encoding.json` | schema version, UTC timestamp, source parquet, `categorical_features` (column -> train levels), `model_cols` in exact order, dummy dtype, notes | a scoring job reproducing the matrix |

Both files are read back and checked before the summary prints. This section and §10 are both safe to
re-run in place, in either order: §11 drops the dummies it left on `out` and recomputes them, and §10
excludes them from both its numeric coercion and its parquet, so that file stays the readable one.

### 11.1 / 11.2. What to encode, and where the levels come from

Two things to read in the output: the four-column set with its level and null counts, and the report
of any value that occurs **outside** `train` but is not a train level. Those rows will encode to
`__other`, so they are printed as a WARNING block rather than left to be inferred from an all-zero
column.

Nothing here mutates `out` - the values are read as strings into `CAT_STR` - so the report is
identical on a second run.

In [ ]:
# ---------------------------------------------------------------------------
# 11.1  WHAT TO ENCODE - derived from X_COLS, NEVER from §10's CATEGORICAL.
# ---------------------------------------------------------------------------
# §10's CATEGORICAL = DESCRIPTIVE + MMSC_CATEGORICAL + ["split", "school_group"].
# The last two are in there because parquet needs a dtype for them, NOT because
# they are features. Deriving this list from X_COLS keeps them out. Order follows
# X_COLS, so it is deterministic.
CATEGORICAL_FEATURES = [
    c for c in X_COLS
    if c in out.columns and isinstance(out[c].dtype, pd.CategoricalDtype)
]

# --- the tripwire: one forbidden-set check, keyed by name -> why. ------------
# Every entry is unreachable while X_COLS == FEATURES; it fires only if someone
# widens X_COLS.
_NEVER_A_CATEGORICAL_FEATURE = {
    "split": (
        "target leakage of the split itself - one-hot encoding it puts the "
        "train/test assignment into the feature matrix as split__train / "
        "split__test, every off-train metric looks wonderful, and nothing else "
        "raises. Bookkeeping: it is in §10's CATEGORICAL only because "
        "to_parquet needs a dtype for it."
    ),
    "school_group": (
        "§9's A/B school partition - the other half of the split assignment, so "
        "encoding it is the same leakage as encoding 'split'. Bookkeeping."
    ),
    "school_id": (
        "a ~30k-cardinality KEY that §10 casts to category so parquet accepts "
        "it; one-hot encoding it would emit ~30k columns."
    ),
    Y_COL: (
        "the target. Encoding the label into the feature matrix is total leakage."
    ),
    "product_score": (
        "the Retention Health Score this model is scored AGAINST. A PASSTHROUGH "
        "baseline, in EXCLUDE_EXACT, and it must never reach X_COLS."
    ),
}
_NEVER_A_CATEGORICAL_FEATURE.update({
    c: ("a passthrough / baseline column - selected to ride along in the frame, "
        "never an input. In EXCLUDE_EXACT.")
    for c in (set(PASSTHROUGH) | set(BASELINE))
    if c not in _NEVER_A_CATEGORICAL_FEATURE
})
if ORG_COL:
    _NEVER_A_CATEGORICAL_FEATURE.setdefault(ORG_COL, (
        "the organization identifier - §12e's diagnostic key. High-cardinality, "
        "and it carries the org-level subscription outcome this model is trying "
        "to predict."
    ))

_cf_forbidden = [c for c in CATEGORICAL_FEATURES
                 if c in _NEVER_A_CATEGORICAL_FEATURE]
assert not _cf_forbidden, (
    f"column(s) that must NEVER be a feature reached CATEGORICAL_FEATURES "
    f"{_cf_forbidden} - so §11.3 would one-hot encode them:\n"
    + "\n".join(f"    {c}: {_NEVER_A_CATEGORICAL_FEATURE[c]}"
                for c in _cf_forbidden)
    + "\n  CATEGORICAL_FEATURES is derived as X_COLS x category-dtype, so the "
      "only way in is via X_COLS. Check what widened it."
)

print(f"CATEGORICAL_FEATURES: {len(CATEGORICAL_FEATURES)} column(s), derived as "
      "X_COLS x category-dtype\n")
print(f"  {'column':<36}{'levels (all rows)':>19}{'nulls':>12}{'null %':>9}")
for c in CATEGORICAL_FEATURES:
    _n_null = int(out[c].isna().sum())
    print(f"  {c:<36}{out[c].nunique(dropna=True):>19}{_n_null:>12,}"
          f"{_n_null / len(out):>8.2%}")

_not_encoded = [c for c in CATEGORICAL if c in out.columns
                and c not in CATEGORICAL_FEATURES]
print(f"\n  category dtype but NOT encoded (not in X_COLS): {_not_encoded}")
print("  -> bookkeeping / keys. 'split' and 'school_group' are carried into the")
print("     encoded file in 11.4 for cell selection, but never as features.")

# ---------------------------------------------------------------------------
# 11.2  LEVELS COME FROM THE TRAIN CELL ONLY.
# ---------------------------------------------------------------------------
# Sorted for determinism - the column order of the matrix is part of the
# contract a scoring job replays.
_train_mask = out["split"] == "train"
assert _train_mask.any(), (
    "no rows in the 'train' split, so there are no levels to freeze. Re-run §9 "
    "(the split cell) before this one."
)


def _cat_strings(col: str) -> pd.Series:
    """The column's values as strings, with nulls left as NaN.

    NOT .astype(str) on the whole column: that renders NaN as the literal
    string "nan", which would then become a real level and quietly corrupt a
    column whose missingness is meaningful. Same null-safe pattern as §10's
    _str_category.
    """
    s = out[col].astype(object)
    return s.where(s.isna(), s.astype(str))


# Read once, reused by 11.3. Nothing here mutates `out`, which is what makes the
# unseen-value report below identical on a second run instead of empty.
CAT_STR = {c: _cat_strings(c) for c in CATEGORICAL_FEATURES}

CAT_LEVELS = {
    c: sorted({str(v) for v in CAT_STR[c][_train_mask].dropna().unique()})
    for c in CATEGORICAL_FEATURES
}

print(f"\ntrain levels frozen from {int(_train_mask.sum()):,} train rows:")
for c in CATEGORICAL_FEATURES:
    print(f"  {c:<36}{len(CAT_LEVELS[c]):>3}  {CAT_LEVELS[c]}")

# --- values that occur OUTSIDE train but are not train levels ---------------
# These get no level column, so they land in c__other. Reported per column with
# row counts and the splits they occur in.
OUTSIDE_TRAIN_LEVELS = {}
for c in CATEGORICAL_FEATURES:
    _s = CAT_STR[c]
    _outside = _s[~_train_mask].dropna()
    _unseen = _outside[~_outside.isin(CAT_LEVELS[c])]
    if len(_unseen):
        _entries = []
        for _lvl, _n in _unseen.value_counts().items():
            _m = (~_train_mask) & _s.eq(_lvl)
            _entries.append({
                "level": str(_lvl),
                "rows": int(_n),
                "splits": sorted({str(s) for s in
                                  out.loc[_m, "split"].astype(object).unique()}),
            })
        OUTSIDE_TRAIN_LEVELS[c] = _entries

if OUTSIDE_TRAIN_LEVELS:
    _unseen_rows = sum(e["rows"] for v in OUTSIDE_TRAIN_LEVELS.values() for e in v)
    print("\n" + "!" * 72)
    print("WARNING: value(s) occur OUTSIDE the train cell but are not train levels,")
    print("so those rows encode to <column>__other - a single bucket shared by every")
    print("such value:")
    for c, _entries in OUTSIDE_TRAIN_LEVELS.items():
        print(f"    {c}:")
        for _e in _entries:
            print(f"        {_e['level']!r:<32} {_e['rows']:>8,} rows "
                  f"({_e['rows'] / len(out):.3%})  in splits {_e['splits']}")
    print(f"  {_unseen_rows:,} row(s) affected in total.")
    print("  If a level shows up here with real volume, extend the train window or")
    print("  collapse the level into a documented bucket BEFORE this cell - never")
    print("  widen CAT_LEVELS from the off-train cells.")
    print("!" * 72)
else:
    print("\nno unseen values: every non-null value outside the train cell is also a "
          "train level, so __other will be all-zero everywhere.")

### 11.3. Encode

Writes one indicator per train level, plus `__other` and `__isna`, onto `out`. Any dummy a previous
run left behind is dropped first, so `out` cannot accumulate columns and `MODEL_COLS` cannot drift.

Three assertions, in the order they can go wrong: no two levels of one column slug to the same name
(and no level slugs to the empty string or to the reserved `other` / `isna`); no generated name
collides with a column already in `out`, and none is generated twice across source columns; and every
row's indicators for a column sum to exactly 1.

The output then reports which `__other` / `__isna` / level columns came out all-zero on this extract.
They are kept regardless, so the column set is identical at scoring time.

In [ ]:
# ---------------------------------------------------------------------------
# 11.3  ENCODE - one indicator per train level, plus __other and __isna.
# ---------------------------------------------------------------------------
DUMMY_SEP    = "__"
OTHER_SUFFIX = f"{DUMMY_SEP}other"    # non-null, but not a train level
ISNA_SUFFIX  = f"{DUMMY_SEP}isna"     # null
DUMMY_DTYPE  = "uint8"
RESERVED_SLUGS = {"other", "isna"}    # a level may not slug to either


def slug(value) -> str:
    """Level string -> column-name fragment.

    Lowercase, every run of non-alphanumeric characters becomes a single '_',
    then leading/trailing '_' are stripped:

        '10-39 small'       -> '10_39_small'
        '1. New'            -> '1_new'
        'SMB Multisite 2-9' -> 'smb_multisite_2_9'
        'UNITED STATES'     -> 'united_states'

    The result has to be safe for parquet, for XGBoost's sklearn API (which
    rejects '[', ']' and '<' in feature names) and for a human reading a SHAP
    driver list. Because it is lossy, the caller ASSERTS that no two levels of
    one column produce the same slug.
    """
    return re.sub(r"[^0-9a-z]+", "_", str(value).lower()).strip("_")


# --- re-runnability: drop dummies left by a previous run of this section -----
# Recompute from the raw categoricals rather than appending, so running 11.3
# twice cannot duplicate a column or change MODEL_COLS. Anything that is a real
# feature / key / passthrough is protected from this drop by name.
_PROTECTED_FROM_DROP = (set(X_COLS) | set(KEYS) | set(PASSTHROUGH)
                        | set(ID_PASSTHROUGH) | {Y_COL, "split", "school_group"})
_dummy_prefixes = tuple(f"{c}{DUMMY_SEP}" for c in CATEGORICAL_FEATURES)
_stale = [c for c in out.columns
          if c.startswith(_dummy_prefixes) and c not in _PROTECTED_FROM_DROP]
if _stale:
    out.drop(columns=_stale, inplace=True)
    print(f"re-run: dropped {len(_stale)} dummy column(s) from a previous pass "
          "before recomputing")

# --- slug the levels, and refuse a collision --------------------------------
CAT_SLUGS = {}
for c in CATEGORICAL_FEATURES:
    _seen = {}
    for _lvl in CAT_LEVELS[c]:
        _sl = slug(_lvl)
        assert _sl, (
            f"level {_lvl!r} of {c!r} slugs to the EMPTY string - every character "
            "in it is non-alphanumeric, so it cannot become a column name. Rename "
            "or bucket the level upstream."
        )
        assert _sl not in RESERVED_SLUGS, (
            f"level {_lvl!r} of {c!r} slugs to {_sl!r}, which is reserved for this "
            f"column's own bookkeeping indicator {c}{DUMMY_SEP}{_sl}. Two "
            "different facts would share one column. Rename the level upstream."
        )
        assert _sl not in _seen, (
            f"SLUG COLLISION on {c!r}: levels {_seen[_sl]!r} and {_lvl!r} both "
            f"slug to {_sl!r}, so they would share the column "
            f"{c}{DUMMY_SEP}{_sl}. Slugging is lossy (case, punctuation and "
            "whitespace runs all collapse), and a collision must FAIL rather than "
            "silently merge two levels into one column. Rename one level "
            "upstream, or bucket them deliberately."
        )
        _seen[_sl] = _lvl
    CAT_SLUGS[c] = [slug(l) for l in CAT_LEVELS[c]]

# --- the column plan, in the order MODEL_COLS will use ----------------------
ENCODE_PLAN = []
for c in CATEGORICAL_FEATURES:
    ENCODE_PLAN.append({
        "column": c,
        "levels": list(CAT_LEVELS[c]),
        "level_cols": [f"{c}{DUMMY_SEP}{s}" for s in CAT_SLUGS[c]],
        "other_col": f"{c}{OTHER_SUFFIX}",
        "isna_col": f"{c}{ISNA_SUFFIX}",
    })

DUMMY_COLS = [x for _p in ENCODE_PLAN
              for x in _p["level_cols"] + [_p["other_col"], _p["isna_col"]]]

_dupe_dummies = sorted({x for x in DUMMY_COLS if DUMMY_COLS.count(x) > 1})
assert not _dupe_dummies, (
    f"generated dummy name(s) appear twice ACROSS columns: {_dupe_dummies}. Two "
    "source columns produced the same name (e.g. column 'a_b' + level 'c' and "
    f"column 'a' + level 'b{DUMMY_SEP}c'). Rename a level upstream."
)
_collides = sorted(set(DUMMY_COLS) & set(out.columns))
assert not _collides, (
    "generated dummy name(s) collide with column(s) already in `out`: "
    f"{_collides}. Writing them would overwrite real data - a feature, a key or "
    "a passthrough - with an indicator. Rename the level upstream, or change "
    f"DUMMY_SEP ({DUMMY_SEP!r})."
)

# --- compute the three mutually exclusive groups ----------------------------
# level  : value == that train level
# __other: value is non-null but not a train level
# __isna : value is null
# Read from CAT_STR (11.2) - the RAW columns as strings - so this is idempotent.
for _p in ENCODE_PLAN:
    _s = CAT_STR[_p["column"]]
    _isna = _s.isna()
    _known = _s.isin(_p["levels"])
    for _col, _lvl in zip(_p["level_cols"], _p["levels"]):
        out[_col] = _s.eq(_lvl).to_numpy().astype(DUMMY_DTYPE)
    out[_p["other_col"]] = ((~_isna) & (~_known)).to_numpy().astype(DUMMY_DTYPE)
    out[_p["isna_col"]] = _isna.to_numpy().astype(DUMMY_DTYPE)


def assert_one_hot_invariant(frame, plan=None):
    """Assert every row's indicators for each categorical sum to exactly 1.

    The level / __other / __isna groups are mutually exclusive and exhaustive by
    construction, so this single invariant catches a slug collision, a missed
    level and a null-handling bug at once. Reusable at scoring time: hand it the
    matrix a scoring job built and the plan it built it from.
    """
    for _p in (ENCODE_PLAN if plan is None else plan):
        _cols = _p["level_cols"] + [_p["other_col"], _p["isna_col"]]
        _sums = frame[_cols].to_numpy(dtype="int64").sum(axis=1)
        _bad = int((_sums != 1).sum())
        assert _bad == 0, (
            f"{_bad:,} row(s) do NOT sum to exactly 1 across the {len(_cols)} "
            f"indicators of {_p['column']!r} (min {int(_sums.min())}, max "
            f"{int(_sums.max())}). The level / __other / __isna groups must "
            "partition every row: a 0 means no group claimed it (a missed level "
            "or a null-handling bug) and a 2 means two did (a slug collision, or "
            "a level string that is also being counted as __other)."
        )
    return True


assert_one_hot_invariant(out)

# Dummy dtype is asserted in 11.4, together with the float32 dtype of the rest of
# MODEL_COLS, as one exact expected-dtype map.

# --- report -----------------------------------------------------------------
# Per-column shape (levels -> columns) is 11.4's summary table; per-column
# isna=1 is 11.1's null count.
print(f"\n{len(CATEGORICAL_FEATURES)} categorical(s) -> {len(DUMMY_COLS)} indicator "
      f"columns ({DUMMY_DTYPE}), added to `out`")
print("one-hot invariant OK: every row sums to exactly 1 within each column's "
      "indicator group")

# --- which indicators came out all-zero -------------------------------------
_zero_other = [_p["other_col"] for _p in ENCODE_PLAN
               if not out[_p["other_col"]].any()]
_zero_isna = [_p["isna_col"] for _p in ENCODE_PLAN
              if not out[_p["isna_col"]].any()]
_zero_level = [x for _p in ENCODE_PLAN for x in _p["level_cols"]
               if not out[x].any()]

print("\nall-zero on this extract (KEPT, so the schema is stable):")
print(f"  __other all-zero : {len(_zero_other)} of {len(ENCODE_PLAN)}  {_zero_other}")
print(f"  __isna  all-zero : {len(_zero_isna)} of {len(ENCODE_PLAN)}  {_zero_isna}")
if _zero_level:
    print(f"  level all-zero   : {len(_zero_level)}  {_zero_level}")
    print("    (a train level in NO row of the full frame - possible if the split")
    print("     cell changed since CAT_LEVELS was frozen; re-run 11.2)")

# --- __other and __isna are DIFFERENT facts, reported separately ------------
for _p in ENCODE_PLAN:
    _n_other = int(out[_p["other_col"]].sum())
    if _n_other:
        _in_train = int(out.loc[_train_mask, _p["other_col"]].sum())
        print(f"\n{_p['other_col']}: fires on {_n_other:,} row(s), {_in_train:,} of "
              "them in train")
        if _in_train == 0:
            print("  NOTE: 0 in train, so the model fits no effect for this column -")
            print("  those rows score as if the whole categorical were switched off.")
            print("  See 11.2's WARNING for which values land here.")

### 11.4. `MODEL_COLS`, assert, persist

Builds `MODEL_COLS` and the encoded frame, asserts them, and writes the encoded parquet plus the JSON
schema. Their stem is taken from `OUT.stem`, so all three files sort together in `data/raw/`.

The assertions before the write: **no duplicates** in `MODEL_COLS`; **none of** `split`,
`school_group`, `school_id`, the org id, `product_score`, `label` or any raw categorical column is
present; and **one exact dtype per column** - every dummy `uint8`, every surviving `X_COLS` feature
`float32`. After the write, both files are read back: the parquet for shape, column order and feature
dtypes and the one-hot invariant, and the JSON for `model_cols` and `categorical_features`.

In [ ]:
# ---------------------------------------------------------------------------
# 11.4  MODEL_COLS - the authoritative feature list.
# ---------------------------------------------------------------------------
# Non-categorical members of X_COLS in their EXISTING order, then the dummies
# grouped by source column (levels in CAT_LEVELS order, then __other, then
# __isna). Deterministic ordering is not cosmetic: it is what the scoring path
# replays from the JSON schema.
_UNCHANGED_X = [c for c in X_COLS if c not in set(CATEGORICAL_FEATURES)]
MODEL_COLS = _UNCHANGED_X + DUMMY_COLS

# Duplicates: checked once, here. The encoded frame's column list is
# `_carried + MODEL_COLS`, `_carried` is de-duplicated as it is built, and the
# _forbidden check below proves _carried and MODEL_COLS are disjoint - so
# no-dupes here plus disjoint there is exactly no-dupes in the encoded frame.
_model_dupes = sorted({c for c in MODEL_COLS if MODEL_COLS.count(c) > 1})
assert not _model_dupes, (
    f"duplicate entries in MODEL_COLS: {_model_dupes}. A booster fitted on this "
    "list and a matrix selected by it would disagree about width, and the same "
    "duplication reaches the encoded parquet's column list."
)

# Everything that must NOT be a feature, in one place, checked by name.
_FORBIDDEN_IN_MODEL = sorted(
    {"split", "school_group", "school_id", Y_COL, "product_score"}
    | set(PASSTHROUGH) | set(BASELINE) | set(ID_PASSTHROUGH) | set(KEYS)
    | set(CATEGORICAL_FEATURES)
    | ({ORG_COL} if ORG_COL else set())
)
_forbidden = sorted(set(_FORBIDDEN_IN_MODEL) & set(MODEL_COLS))
assert not _forbidden, (
    f"column(s) that must never be features are in MODEL_COLS: {_forbidden}.\n"
    "  split / school_group  -> target leakage of the split assignment itself\n"
    "  school_id / org id    -> keys the tree would memorise\n"
    "  product_score         -> the baseline this model is scored AGAINST\n"
    f"  {Y_COL} -> the target\n"
    "  a raw categorical     -> carried for readability; the model reads its "
    "dummies instead"
)

print(f"MODEL_COLS   {len(MODEL_COLS)} = {len(_UNCHANGED_X)} non-categorical X_COLS "
      f"+ {len(DUMMY_COLS)} dummies")
print(f"X_COLS       {len(X_COLS)}  ({len(CATEGORICAL_FEATURES)} categorical(s) "
      f"replaced by {len(DUMMY_COLS)} indicator columns)")
print("raw categoricals stay in `out` and in the encoded file for readability - "
      "they are simply not in MODEL_COLS")

# ---------------------------------------------------------------------------
# The encoded frame: keys, label, cell selection, passthroughs, the RAW
# categoricals (readability / debugging), then all of MODEL_COLS.
# ---------------------------------------------------------------------------
NON_FEATURE_CARRIED = ["split", "school_group"]

_carried = []
for c in (KEYS + [Y_COL] + NON_FEATURE_CARRIED + list(PASSTHROUGH)
          + list(CATEGORICAL_FEATURES)):
    if c in out.columns and c not in _carried:
        _carried.append(c)

ENCODED_COLS = _carried + MODEL_COLS
enc = out[ENCODED_COLS]

assert len(enc) == len(out), (
    f"{len(enc):,} rows in the encoded frame vs {len(out):,} in §10's - encoding "
    "widens a dataset, it must not change the row count."
)
assert list(enc.columns)[-len(MODEL_COLS):] == MODEL_COLS, (
    "MODEL_COLS is not the tail of the encoded frame's column list, in order - a "
    "booster fitted on one and scored on the other mis-aligns silently"
)

# --- dtypes AS BUILT: one exact expected-dtype map -------------------------
# This is the "wrong dtype when built" half of the dtype coverage; the round-trip
# check further down is the "wrong dtype after the parquet round trip" half.
ENCODED_DTYPES = {DUMMY_DTYPE, "float32"}
_EXPECTED_MODEL_DTYPE = {c: DUMMY_DTYPE for c in DUMMY_COLS}
_EXPECTED_MODEL_DTYPE.update({c: "float32" for c in _UNCHANGED_X})

_bad_enc_dtypes = [(c, str(enc[c].dtype), _EXPECTED_MODEL_DTYPE[c])
                   for c in MODEL_COLS
                   if str(enc[c].dtype) != _EXPECTED_MODEL_DTYPE[c]]
assert not _bad_enc_dtypes, (
    f"MODEL_COLS column(s) do not have the dtype they were built with "
    f"(allowed: {sorted(ENCODED_DTYPES)}, one exact expectation per column):\n"
    + "\n".join(f"    {c:<52} is {d}, expected {e}"
                for c, d, e in _bad_enc_dtypes)
    + "\n  A `category` dtype here means CATEGORICAL_FEATURES missed a category "
      "column inside X_COLS - XGBoost either refuses it or silently reads "
      "integer codes. An `object` dtype produces the opaque pyarrow error that "
      f"names a Python type rather than a column. A dummy that is not "
      f"{DUMMY_DTYPE} means something re-cast the indicators - typically "
      "re-running §10 on its own after §11."
)

# ---------------------------------------------------------------------------
# Write: the encoded parquet, and the JSON encoding schema.
# ---------------------------------------------------------------------------
# Stem TAKEN FROM §10's OUT, not rebuilt from the same constants, so the three
# files cannot be named apart and still sort together in data/raw/.
ENCODED_STEM = OUT.stem
OUT_ENCODED  = RAW_DIR / f"{ENCODED_STEM}_encoded.parquet"
OUT_ENCODING = RAW_DIR / f"{ENCODED_STEM}_encoding.json"

enc.to_parquet(OUT_ENCODED, index=False)

ENCODING_SCHEMA_VERSION = 1

# The sidecar is THE contract: everything a scoring job needs to rebuild this
# exact matrix from raw rows, and nothing it has to guess.
ENCODING_SCHEMA = {
    "schema_version": ENCODING_SCHEMA_VERSION,
    "generated_at_utc": pd.Timestamp.now(tz="UTC").strftime("%Y-%m-%dT%H:%M:%SZ"),
    "source_parquet": OUT.name,
    "encoded_parquet": OUT_ENCODED.name,
    "levels_frozen_from_split": "train",
    "dummy_separator": DUMMY_SEP,
    "other_suffix": OTHER_SUFFIX,
    "isna_suffix": ISNA_SUFFIX,
    "dummy_dtype": DUMMY_DTYPE,
    "categorical_features": {c: list(CAT_LEVELS[c]) for c in CATEGORICAL_FEATURES},
    "model_cols": list(MODEL_COLS),
    "keys": list(KEYS),
    "y_col": Y_COL,
    "non_feature_carried": list(NON_FEATURE_CARRIED),
    "passthrough": list(PASSTHROUGH),
    "unseen_values_outside_train": OUTSIDE_TRAIN_LEVELS,
    "scaling": "none - trees are threshold-based, so monotonic rescaling is a no-op",
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "notes": (
        "One-hot encoding of the categorical FEATURES only. Levels are frozen "
        "from the 'train' split. Per column the three indicator groups are "
        "mutually exclusive and exhaustive and sum to exactly 1 on every row: "
        "one column per train level, plus '<column>__other' for a value that is "
        "non-null but NOT a train level (an unseen level maps to __other), plus "
        "'<column>__isna' for a null. __other and __isna are always emitted, "
        "even when all-zero, so the column set is identical at scoring time. "
        "Replay 'model_cols' in exact order; the raw categorical columns are "
        "carried in the encoded parquet for readability and are not features."
    ),
}

with OUT_ENCODING.open("w") as _f:
    json.dump(ENCODING_SCHEMA, _f, indent=2)
    _f.write("\n")

# --- round-trip both --------------------------------------------------------
_back_enc = pd.read_parquet(OUT_ENCODED)
assert _back_enc.shape == enc.shape, (
    f"parquet round trip changed the shape: {enc.shape} -> {_back_enc.shape}"
)
assert list(_back_enc.columns) == list(enc.columns), (
    "parquet round trip changed the column order"
)
# The "wrong dtype AFTER the round trip" half.
_rt_dtypes = [(c, str(enc[c].dtype), str(_back_enc[c].dtype)) for c in MODEL_COLS
              if str(_back_enc[c].dtype) != str(enc[c].dtype)]
assert not _rt_dtypes, (
    "parquet round trip changed feature dtype(s) - a uint8 that comes back as "
    "something else changes what XGBoost is handed:\n"
    + "\n".join(f"    {c:<52} {a} -> {b}" for c, a, b in _rt_dtypes)
)
assert_one_hot_invariant(_back_enc)

# --- do the JSON, the in-memory frame and the round-tripped frame agree? -----
# Three artefacts, three statements: the MODEL_COLS-is-the-tail assertion above
# ties the in-memory frame to the list, the column-order assertion just above
# ties the round-tripped frame to the in-memory one, and the one below ties the
# JSON to the list. JSON == round-tripped tail then follows.
_schema_back = json.loads(OUT_ENCODING.read_text())
assert _schema_back["model_cols"] == MODEL_COLS, (
    "the JSON schema's model_cols does not match MODEL_COLS - a scoring job "
    "reading the sidecar would build a mis-aligned matrix. Because the encoded "
    "frame's tail and the round-tripped column order are asserted above, this "
    "mismatch means the sidecar disagrees with BOTH frames."
)
assert _schema_back["categorical_features"] == {
    c: list(CAT_LEVELS[c]) for c in CATEGORICAL_FEATURES
}, "the JSON schema's categorical_features do not match CAT_LEVELS"
print(f"\nround trip OK: parquet shape {_back_enc.shape}, column order and feature "
      "dtypes unchanged, one-hot invariant holds on the")
print("reloaded frame, and the sidecar's model_cols / categorical_features match")

# ---------------------------------------------------------------------------
# Summary.
# ---------------------------------------------------------------------------
print("\nlevels per categorical (frozen from `train`):")
for _p in ENCODE_PLAN:
    print(f"  {_p['column']:<36}{len(_p['levels']):>3} level(s) -> "
          f"{len(_p['level_cols']) + 2:>3} columns "
          f"({len(_p['level_cols'])} + __other + __isna)")
print(f"\nlen(MODEL_COLS)         {len(MODEL_COLS)}")
print(f"encoded frame           {enc.shape[0]:,} rows x {enc.shape[1]} cols")
print(f"in memory               {enc.memory_usage(deep=True).sum() / 1e6:.1f} MB")
print(f"on disk                 {OUT_ENCODED.stat().st_size / 1e6:.1f} MB")
print(f"\nwrote  {OUT_ENCODED.name}")
print(f"       {OUT_ENCODING.name}   "
      f"({OUT_ENCODING.stat().st_size / 1e3:.1f} KB)")
print(f"beside {OUT.name}   <- §10, still the canonical readable dataset")
print("\nper-cell composition is unchanged by encoding - see §9's table.")
print("\nNOTE: `out` now carries the dummy columns. Both sections are safe to re-run")
print("in place - §10 excludes the dummies from its numeric coercion and from its")
print("parquet, so that file stays the readable one; §11 drops its own dummies and")
print("recomputes them.")

In [ ]:
print(f"Total positives: {out['label'].sum():,}")
print(f"Positive rate: {out['label'].mean():.2%}")
print(f"\nOriginal (before fix): 24,285 positives, 3.04% rate")

In [ ]:
print(f"LABEL_HORIZON_MONTHS currently set to: {LABEL_HORIZON_MONTHS}")
print(f"OBS_END: {OBS_END}")

---

## 12. Appendix. Data checks (run when something looks wrong)

**Optional.** Nothing in §1-§11 depends on anything defined in this section, so the main flow runs
top-to-bottom without it. Each cell reads only §3's `run_query`, §5's `TABLES`, §6's constants and
the `out` frame.

| cell | the question it answers |
| --- | --- |
| 12a | Which tables do I actually have `SELECT` on, and how many rows does each hold? |
| 12b | What does `school_country` actually contain, and how much does the country filter keep? |
| 12c | What is the label made of, and how much real churn can it never see? |
| 12d | What did the segment and government filters cost? |
| 12e | Do `test`-cell schools share an organization with `train`-cell schools? |

**Run 12b before §8** if the extract returns zero rows: a country filter that matches nothing returns
zero rows with no SQL error, and 12b is what shows you the stored values.

### 12a. Access and row counts

Which of the four tables in `TABLES` you can `SELECT` from, and how big each is. A `permission
denied` here means you need a GRANT - note the table name and ask in `#ask-dataeng`.

In [ ]:
for name, table in TABLES.items():
    try:
        n = run_query(f"select count(*) as n from {table}").iloc[0, 0]
        print(f"{name:<16} {table:<58} {n:>12,} rows")
    except Exception as e:
        print(f"{name:<16} {table:<58} FAILED: {type(e).__name__}: {e}")

### 12b. What does `school_country` actually contain?

The distinct stored values of `school_country` over the observation window, with the row count each
carries and whether `POPULATION_COUNTRIES` keeps or drops it. Run this before §8 if the extract
returns zero rows.

In [ ]:
# Reads §6's constants: OBS_START / OBS_END / PREMIUM_FILTER /
# POPULATION_COUNTRIES / norm_country. No duplicated window literal.
COUNTRY_DISCOVERY_SQL = f"""
select
      school_country
    , count(*)                  as row_count
    , count(distinct school_id) as schools
from {TABLES['usage_trends']}
where dim_month >= date '{OBS_START}'
  and dim_month <  date '{OBS_END}'
  and {PREMIUM_FILTER}
group by 1
order by row_count desc
"""

country_discovery = run_query(COUNTRY_DISCOVERY_SQL)

# Same normalisation the SQL predicate uses: upper(trim(...)).
_norm = country_discovery["school_country"].map(norm_country)
_keep = _norm.isin(POPULATION_COUNTRIES)

_total = int(country_discovery["row_count"].sum())
_kept_rows = int(country_discovery.loc[_keep, "row_count"].sum())

print(f"POPULATION_COUNTRIES (normalised) = {POPULATION_COUNTRIES}\n")
print(f"{'':<6}{'stored value':<28}{'normalised':<28}{'rows':>14}")
for _, _r in country_discovery.iterrows():
    _v = _r["school_country"]
    _n = norm_country(_v)
    _mark = "KEEP" if _n in POPULATION_COUNTRIES else "DROP"
    print(f"{_mark:<6}{str(_v):<28}{str(_n):<28}{int(_r['row_count']):>14,}")

_pct = f"  ({_kept_rows / _total:.1%})" if _total else "  (n/a - no rows)"
print(f"\npremium school-months in window {_total:>14,}")
print(f"KEPT by the country filter      {_kept_rows:>14,}{_pct}")
print(f"DROPPED by the country filter   {_total - _kept_rows:>14,}")

_unused_accepted = [c for c in POPULATION_COUNTRIES if c not in set(_norm.dropna())]
if _unused_accepted:
    print(f"\nNOTE: accepted value(s) that do NOT occur in this window: {_unused_accepted}")

_dropped_values = sorted({str(v) for v in _norm[~_keep].dropna().unique()})
if _dropped_values:
    print(f"\nstored value(s) the filter DROPS: {_dropped_values}")
    print("  A US or Canada spelling appearing here would be new - add it to")
    print("  POPULATION_COUNTRIES deliberately rather than widening the predicate.")

if _kept_rows == 0:
    print("\n" + "!" * 72)
    print("WARNING: POPULATION_COUNTRIES matches NOTHING in this window.")
    print("The §8 extract would return ZERO rows - silently, with no SQL error.")
    print("Fix POPULATION_COUNTRIES to the stored values listed above BEFORE running")
    print("the extract, then re-run §6's configuration cell and §8.")
    print("!" * 72)

country_discovery

### 12c. What the label is made of, and what it cannot see

Two queries. The first breaks the churn-case table down by `summarized_request_type`, showing what
the request-type filter and the `is_target_population` gate each remove. The second restricts to
reportable **churns** and reports how many carried a cancellation request this label counts and how
many did not - involuntary, org-change, or never ticketed.

Both window on `coalesce(case_created_date, churned_date)`, spelled out in full, because a caseless
churn row has no case date and those rows are exactly what is being measured here.

In [ ]:
# --- half 1: what the request-type filter removes --------------------------
EVENT_MIX_SQL = f"""
select
      coalesce(summarized_request_type, '(none)')            as summarized_request_type
    , count(*)                                              as cases
    , sum(case when is_reportable_request then 1 else 0 end) as reportable_requests
    , sum(case when is_reportable_churn   then 1 else 0 end) as reportable_churns
    , sum(case when is_target_population  then 1 else 0 end) as in_target_population
    , sum(case when coalesce(is_reportable_request, false)
                and is_target_population
                and case_created_date is not null
               then 1 else 0 end)                            as label_qualifying
    , sum(case when case_id is null       then 1 else 0 end) as rows_without_case
from {TABLES['churn_cases']}
-- Explicit coalesce: an audit of what the label CANNOT see has to give the
-- caseless churn rows a date.
where coalesce(case_created_date, churned_date) >= date '{OBS_START}'
  and coalesce(case_created_date, churned_date) <  date '{OBS_END}'
group by 1
order by cases desc
"""

event_mix = run_query(EVENT_MIX_SQL)
_kept = event_mix["summarized_request_type"].isin(CANCEL_REQUEST_TYPES)

print(f"cases in window                : {event_mix['cases'].sum():,}")
print(f"reportable requests            : {event_mix['reportable_requests'].sum():,}")
print(f"  counted as positive          : {event_mix.loc[_kept, 'reportable_requests'].sum():,}")
print(f"  dropped (not a cancelation request, incl. failed payments): "
      f"{event_mix.loc[~_kept, 'reportable_requests'].sum():,}")
print(f"\nafter the full label predicate (flag + type + is_target_population + dated):")
print(f"  label-qualifying events        : "
      f"{event_mix.loc[_kept, 'label_qualifying'].sum():,}")
print(f"  removed by the is_target_population gate: "
      f"{event_mix.loc[_kept, 'reportable_requests'].sum() - event_mix.loc[_kept, 'label_qualifying'].sum():,}")

# --- half 2: how much churn the label can never see ------------------------
# Same window, restricted to reportable CHURN - actual departures, not requests.
CHURN_BLIND_SPOT_SQL = f"""
select
      coalesce(summarized_request_type, '(none)')             as summarized_request_type
    , coalesce(churn_classification, '(none)')                as churn_classification
    , count(*)                                                as churns
    , sum(case when case_id is null then 1 else 0 end)         as churns_without_case
from {TABLES['churn_cases']}
where is_reportable_churn
  -- Explicit coalesce again - see the note on the query above.
  and coalesce(case_created_date, churned_date) >= date '{OBS_START}'
  and coalesce(case_created_date, churned_date) <  date '{OBS_END}'
group by 1, 2
order by churns desc
"""

churn_blind_spot = run_query(CHURN_BLIND_SPOT_SQL)

_churn_total = int(churn_blind_spot["churns"].sum()) if len(churn_blind_spot) else 0

print("\n" + "=" * 72)
print("THE LABEL'S BLIND SPOT - reportable churns in the same window")
print("=" * 72)

if _churn_total == 0:
    print("No reportable churns returned for this window. Either the window is wrong")
    print("or is_reportable_churn is not populated - the blind-spot figures below")
    print("cannot be computed, so do not read this cell as 'no blind spot'.")
else:
    _by_type = (churn_blind_spot.groupby("summarized_request_type", dropna=False)["churns"]
                                .sum().sort_values(ascending=False))
    print("\nreportable churns by summarized_request_type:")
    for _v, _n in _by_type.items():
        _flag = "COUNTABLE" if _v in CANCEL_REQUEST_TYPES else "invisible"
        print(f"  {_flag:<10} {str(_v):<38} {_n:>10,}  ({_n / _churn_total:>6.1%})")

    _by_class = (churn_blind_spot.groupby("churn_classification", dropna=False)["churns"]
                                 .sum().sort_values(ascending=False))
    print("\nreportable churns by churn_classification:")
    for _v, _n in _by_class.items():
        print(f"  {str(_v):<49} {_n:>10,}  ({_n / _churn_total:>6.1%})")

    # The four counts to quote when someone asks how big the gap is.
    _by_type_d = {str(k): int(v) for k, v in _by_type.items()}
    _no_case     = int(churn_blind_spot["churns_without_case"].sum())
    _no_request  = _by_type_d.get("No Request", 0)
    _failed_pay  = _by_type_d.get("Failed Payment", 0)
    _capturable  = sum(_by_type_d.get(t, 0) for t in CANCEL_REQUEST_TYPES)

    print(f"\nof {_churn_total:,} reportable churns in the window:")
    for _lbl, _n in (
        ("case_id is null (no ticket at all)",                _no_case),
        ("summarized_request_type = 'No Request'",             _no_request),
        ("summarized_request_type = 'Failed Payment'",         _failed_pay),
        ("captured by the label's Cancelation Request types",  _capturable),
    ):
        print(f"  {_lbl:<52} {_n:>10,}  ({_n / _churn_total:>6.1%})")

    _countable = _capturable
    _invisible = _churn_total - _countable

    print("\n" + "-" * 72)
    print(f"Of {_churn_total:,} reportable churns between {OBS_START} and {OBS_END}:")
    print(f"  {_countable:>10,} ({_countable / _churn_total:>6.1%}) carried a cancelation request "
          "this label counts")
    print(f"  {_invisible:>10,} ({_invisible / _churn_total:>6.1%}) did NOT - involuntary, "
          "org-change, or never ticketed")
    print("-" * 72)
    print("Recall measured against THIS label is not recall against churn.")

churn_blind_spot

### 12d. Population audit - segments and government

What the segment filter and `GOV_FILTER` cost, over the same window and premium filter §12b uses.
`excluded_segment_rows` large relative to `kept_rows` is worth checking against the warehouse's
segment strings - a renamed segment silently shrinks the population rather than erroring, so a
`POPULATION_SEGMENTS` entry that occurs nowhere prints an explicit `WARNING`. Country accounting is
§12b's job.

In [ ]:
POP_AUDIT_SQL = f"""
select
      customer_segment
    , count(*)                                as row_count
    , count(distinct school_id)               as schools
    , sum(case when is_gov then 1 else 0 end) as gov_rows
from {TABLES['usage_trends']}
where dim_month >= date '{OBS_START}'
  and dim_month <  date '{OBS_END}'
  and {PREMIUM_FILTER}
group by 1
order by row_count desc
"""

pop_audit = run_query(POP_AUDIT_SQL)

_seg_ok = pop_audit["customer_segment"].isin(POPULATION_SEGMENTS)

_total = pop_audit["row_count"].sum()
print(f"premium school-months in window      {_total:>12,}"
      "   <- §12b's denominator, before segment/gov/country")
print(f"kept_rows (segment ok)               {pop_audit.loc[_seg_ok, 'row_count'].sum():>12,}")
print(f"excluded_segment_rows                {pop_audit.loc[~_seg_ok, 'row_count'].sum():>12,}")
print(f"\ngov rows inside the segment-kept population (all filtered out by "
      f"GOV_FILTER): {pop_audit.loc[_seg_ok, 'gov_rows'].sum():,}")

# row_count only: `schools` is a distinct count per cell and so is not additive
# across groups.
print("\nby customer_segment:")
for _v, _n in pop_audit.set_index("customer_segment")["row_count"].items():
    print(f"  {'keep' if _v in POPULATION_SEGMENTS else 'DROP'}  {str(_v):<24} {_n:>12,}")

_unmatched_seg = set(POPULATION_SEGMENTS) - set(pop_audit["customer_segment"].dropna())
if _unmatched_seg:
    print(f"\nWARNING: segment string(s) not found in the warehouse: {_unmatched_seg}")
    print("         A renamed segment silently SHRINKS the population instead of")
    print("         erroring, so reconcile POPULATION_SEGMENTS against the list above")
    print("         before trusting any count downstream.")

print("\ncountry accounting: §12b")

pop_audit

### 12e. Multisite / organization leakage

How many `test`-cell schools share an organization with a `train`-cell school, as a count and as a
share of `test` schools. Schools in one organization are not independent observations - subscription
status is org-level and a cancellation case fans out to siblings - so a `school_id` split can leave
siblings straddling it.

`ORG_COL` is probed from the usage-trends catalog in §7 and is a **passthrough** - it is in
`EXCLUDE_EXACT`, so it can never be trained on. If the catalog has no org column, this cell prints
one warning line and the diagnostic goes unmeasured; it never fails the run.

In [ ]:
if not ORG_COL or ORG_COL not in out.columns:
    print("WARNING: no organization identifier in the usage-trends catalog "
          f"(probed {ORG_CANDIDATES}).")
    print("         Multisite / sibling-school leakage across the split is therefore")
    print("         UNMEASURED. Any test-cell metric may be optimistic by an unknown")
    print("         amount. Re-run this cell if an org column is added to the model.")
else:
    _org = out[[ORG_COL, "school_id", "split"]].dropna(subset=[ORG_COL])
    _train_orgs = set(_org.loc[_org["split"] == "train", ORG_COL])
    _test = _org[_org["split"] == "test"]
    _test_schools = _test["school_id"].nunique()
    _straddling = _test.loc[_test[ORG_COL].isin(_train_orgs), "school_id"].nunique()
    _straddling_orgs = _test.loc[_test[ORG_COL].isin(_train_orgs), ORG_COL].nunique()

    print(f"org identifier in use: {ORG_COL}")
    print(f"  orgs represented in train : {len(_train_orgs):,}")
    print(f"  schools in test           : {_test_schools:,}")
    print(f"  ...sharing an org with a train school: {_straddling:,} "
          f"({_straddling / max(_test_schools, 1):.2%} of test schools, "
          f"across {_straddling_orgs:,} orgs)")

    _missing_org = int(out[ORG_COL].isna().sum())
    if _missing_org:
        print(f"  ({_missing_org:,} rows have a null {ORG_COL} and are excluded from "
              "this count - they may hide further straddle)")

    if _straddling:
        print("\n  -> test is NOT org-disjoint from train. Quote this number alongside")
        print("     test metrics, and consider grouping the split on the org id in v2.")
    else:
        print("\n  -> test is org-disjoint from train on this extract. Nothing to correct.")

In [ ]:
norm_cols = sorted(c for c in USAGE_COLS if c.startswith("norm_"))
print(f"{len(norm_cols)} norm_ columns found:")
for c in norm_cols:
    print(f"  {c}")

In [ ]:
customers_catalog = describe_table(TABLES["customers"])
print(customers_catalog["column_name"].tolist())

In [ ]:
org_data = run_query("""
    select
          school_id
        , organization_id
        , is_org_multisite
        , num_schools_in_org
    from analytics_mart_customer.mart_school_customers
""")
print(f"{len(org_data):,} rows pulled")
print(org_data.head())
print(f"\nnulls: {org_data.isna().sum()}")

In [ ]:
org_data.to_parquet(RAW_DIR / "org_lookup.parquet", index=False)
print("saved org_lookup.parquet")

In [ ]:
program_data = run_query("""
    select
          school_id
        , program_type
        , is_program_type_target
    from analytics_mart_customer.mart_school_customers
""")

print(program_data["program_type"].value_counts(dropna=False))
print()
print(program_data["is_program_type_target"].value_counts(dropna=False))

In [ ]:
import pandas as pd
readable = pd.read_parquet(RAW_DIR / "churn_v1_is_reportable_request_202401_202605_h3.parquet")
print(f"loaded {len(readable):,} rows")

In [ ]:
readable_with_program = readable.merge(
    program_data[["school_id", "program_type"]], on="school_id", how="left"
)
print(f"Match rate: {readable_with_program['program_type'].notna().mean():.1%}")
print(readable_with_program["program_type"].value_counts(dropna=False))

In [ ]:
readable_with_program["month_num"] = pd.to_datetime(readable_with_program["dim_month"]).dt.month

camps = readable_with_program[readable_with_program["program_type"] == "camp"]
non_camps = readable_with_program[readable_with_program["program_type"] == "preschool_or_child_care"]

print("Camps - avg active students by month:")
print(camps.groupby("month_num")["product_active_students"].mean())
print(f"\nNon-camps (preschool/childcare) - avg active students by month:")
print(non_camps.groupby("month_num")["product_active_students"].mean())

In [ ]:
program_data.to_parquet(RAW_DIR / "program_type_lookup.parquet", index=False)
print("saved program_type_lookup.parquet")

In [ ]:
customers_catalog = describe_table(TABLES["customers"])
print(f"{len(customers_catalog)} columns in {TABLES['customers']}")

customers_catalog.to_parquet(RAW_DIR / "customers_catalog.parquet", index=False)
print("saved")

In [ ]:
# Back in dataset_creation.ipynb - list all tables in the schema
all_tables = run_query("""
    select table_name
    from svv_tables
    where table_schema = 'analytics_mart_customer'
    order by table_name
""")
print(all_tables)

In [ ]:
print(sorted(USAGE_COLS))

In [ ]:
# In dataset_creation.ipynb
for table_key in ["mart_school_features", "rpt_school_cohort_monthly", "tmp_mart_school_customers_month_history"]:
    full_name = f"analytics_mart_customer.{table_key}"
    try:
        cols = describe_table(full_name)
        print(f"\n=== {table_key} ({len(cols)} columns) ===")
        print(cols["column_name"].tolist())
    except Exception as e:
        print(f"\n{table_key}: FAILED - {e}")

In [ ]:
calendar_check = run_query("""
    select
          school_id
        , month
        , school_calendar_events
        , active_calendar
    from analytics_mart_customer.mart_school_features
    where school_calendar_events > 0
    limit 20
""")
print(calendar_check)

In [ ]:
all_schemas = run_query("""
    select distinct table_schema
    from svv_tables
    where table_schema not like 'pg_%' and table_schema not like 'information_schema'
    order by table_schema
""")
print("SCHEMAS:")
print(all_schemas)

term_search = run_query("""
    select table_schema, table_name
    from svv_tables
    where table_name ilike '%term%' 
       or table_name ilike '%academic%'
       or table_name ilike '%school_year%'
       or table_name ilike '%enrollment_period%'
    order by table_schema, table_name
""")
print("\nTERM/ACADEMIC SEARCH:")
print(term_search)

sfdc_search = run_query("""
    select table_schema, table_name
    from svv_tables
    where table_name ilike '%opportunity%' 
       or table_name ilike '%case%'
       or table_schema ilike '%salesforce%'
       or table_schema ilike '%sfdc%'
    order by table_schema, table_name
""")
print("\nSFDC/CASE SEARCH:")
print(sfdc_search)

In [ ]:
for table in [
    "analytics.groomed_salesforce_cases",
    "analytics_staging.proactive_churn_case_status_tracking_import",
    "analytics_staging.proactive_churn_case_status_tracking_source_import",
    "analytics.stg_salesforce_cases",
]:
    try:
        cols = describe_table(table)
        print(f"\n=== {table} ({len(cols)} columns) ===")
        print(cols["column_name"].tolist())
    except Exception as e:
        print(f"\n{table}: FAILED - {e}")

In [ ]:
usage_check = run_query("""
    select
          count(*) as total_cases
        , count(churn_risk_level_c) as has_risk_level
        , count(at_risk_type_c) as has_at_risk_type
        , count(post_call_notes_c) as has_post_call_notes
        , count(save_tactic_executed_c) as has_save_tactic
        , max(created_date) as most_recent_case
    from analytics.groomed_salesforce_cases
""")
print(usage_check)

In [ ]:
check = name_revenue_lookup.head(20)
print(check[["school_id", "prior_month_billing_net_revenue", "prior_3_month_billing_net_revenue"]])

# Also check basic distribution shape for both
print(name_revenue_lookup[["prior_month_billing_net_revenue", "prior_3_month_billing_net_revenue"]].describe())

In [ ]:
run_query("select current_user, getdate()")

In [ ]:
name_revenue_lookup = run_query("""
    select
          school_id
        , legal_name
        , parse_name
        , prior_month_billing_net_revenue
        , prior_3_month_billing_net_revenue
    from analytics_mart_customer.mart_school_customers
""")
print(f"pulled {len(name_revenue_lookup):,} rows")
print(name_revenue_lookup.head(10))

In [ ]:
print("Null rates:")
print(f"  legal_name: {name_revenue_lookup['legal_name'].isna().mean():.1%}")
print(f"  parse_name: {name_revenue_lookup['parse_name'].isna().mean():.1%}")

print("\nRevenue field distributions:")
print(name_revenue_lookup[["prior_month_billing_net_revenue", "prior_3_month_billing_net_revenue"]].describe())

print("\nHow many schools have ANY non-zero revenue:")
print(f"  prior_month: {(name_revenue_lookup['prior_month_billing_net_revenue'] > 0).sum():,}")
print(f"  prior_3_month: {(name_revenue_lookup['prior_3_month_billing_net_revenue'] > 0).sum():,}")

In [ ]:
# In dataset_creation.ipynb, confirm it saved
name_revenue_lookup.to_parquet(RAW_DIR / "name_revenue_lookup.parquet", index=False)
print("saved")

In [ ]:
LABEL_HORIZON_MONTHS = 6   # was 3

# The notebook has a hardcoded safety check that assumes horizon=3.
# It WILL fail now, on purpose, since changing the horizon changes this boundary.
# Update it to match the horizon=6 case:
_this_month = pd.Timestamp(AS_OF_DATE).normalize().replace(day=1)
LAST_ANSWERABLE_MONTH = _this_month - pd.DateOffset(months=LABEL_HORIZON_MONTHS + 1)
OBS_END = (LAST_ANSWERABLE_MONTH + pd.DateOffset(months=1)).strftime("%Y-%m-%d")

EVAL_START = (pd.Timestamp(year=TRAIN_YEAR, month=12, day=1)
              + pd.DateOffset(months=LABEL_HORIZON_MONTHS + 1))
assert EVAL_START == pd.Timestamp("2025-07-01"), (
    f"EVAL_START derived as {EVAL_START:%Y-%m-%d}, expected 2025-07-01 for horizon=6"
)

print(f"horizon      {LABEL_HORIZON_MONTHS} months")
print(f"window       {OBS_START} -> {OBS_END} (exclusive)")
print(f"last answerable observation month: {LAST_ANSWERABLE_MONTH:%Y-%m}")
print(f"eval starts: {EVAL_START:%Y-%m}")


In [ ]:
case_cols = describe_table("analytics.groomed_salesforce_cases")
priority_case_candidates = [c for c in case_cols["column_name"] if any(
    k in c.lower() for k in ["priority", "rank", "order", "sequence"]
)]
print("Possible priority/rank fields in groomed_salesforce_cases:", priority_case_candidates)

In [ ]:
priority_check = run_query("""
    select
          priority
        , case_ranking_account
        , count(*) as n
    from analytics.groomed_salesforce_cases
    where priority is not null or case_ranking_account is not null
    group by priority, case_ranking_account
    order by n desc
    limit 20
""")
print(priority_check)

In [ ]:
case_type_check = run_query("""
    select
          type
        , record_type_id
        , priority
        , count(*) as n
        , avg(case_ranking_account) as avg_rank
    from analytics.groomed_salesforce_cases
    where case_ranking_account is not null
    group by type, record_type_id, priority
    order by n desc
    limit 20
""")
print(case_type_check)

In [ ]:
# Check whether cases classified as Addressable/Non-Addressable Cancelation Request
# ever carry a reason indicating failed payment, restructure, or permanent closure,
# which would mean these aren't actually excluded, just rare, not impossible

reason_check = run_query("""
    select
          summarized_request_type
        , reason
        , count(*) as n
    from analytics_mart_customer.rpt_school_churn_cases
    where summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
    group by summarized_request_type, reason
    order by n desc
    limit 40
""")
print(reason_check.to_string(index=False))

In [ ]:
churn_cases_cols = describe_table("analytics_mart_customer.rpt_school_churn_cases")
print(churn_cases_cols["column_name"].tolist())

In [ ]:
reason_check = run_query("""
    select
          summarized_request_type
        , cancellation_reason_category
        , is_failed_payment
        , count(*) as n
    from analytics_mart_customer.rpt_school_churn_cases
    where summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
    group by summarized_request_type, cancellation_reason_category, is_failed_payment
    order by n desc
    limit 40
""")
print(reason_check.to_string(index=False))

In [ ]:
reportable_check = run_query("""
    select
          cancellation_reason_category
        , is_reportable_request
        , count(*) as n
    from analytics_mart_customer.rpt_school_churn_cases
    where cancellation_reason_category in ('Failed Payment', 'Business Changes', 'Facility Closed')
    group by cancellation_reason_category, is_reportable_request
    order by n desc
""")
print(reportable_check.to_string(index=False))

In [ ]:
mislabeled_check = run_query("""
    select
          c.cancellation_reason_category
        , count(*) as n_affected_rows
    from analytics_mart_customer.rpt_school_churn_cases c
    where c.is_reportable_request = true
      and c.cancellation_reason_category in ('Failed Payment', 'Business Changes', 'Facility Closed')
    group by c.cancellation_reason_category
""")
print(mislabeled_check)

In [ ]:
real_overlap_check = run_query(f"""
    select
          c.cancellation_reason_category
        , count(*) as n_affected_in_extract
    from analytics_mart_customer.rpt_school_churn_cases c
    where c.is_reportable_request = true
      and c.cancellation_reason_category in ('Failed Payment', 'Business Changes', 'Facility Closed')
      and c.summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
      and c.is_target_population = true
      and c.dim_month >= '{OBS_START}'
      and c.dim_month < '{OBS_END}'
    group by c.cancellation_reason_category
""")
print(real_overlap_check)
print(f"\nTotal affected within your actual extract population: {real_overlap_check['n_affected_in_extract'].sum():,}")

In [ ]:
# Pull the exact schools and event months affected by the exclusion,
# within your actual population and time window
affected_schools_query = run_query(f"""
    select
          c.school_id
        , date_trunc('month', c.case_created_date)::date as event_month
        , c.cancellation_reason_category
    from analytics_mart_customer.rpt_school_churn_cases c
    where c.is_reportable_request = true
      and c.cancellation_reason_category in ('Failed Payment', 'Business Changes', 'Facility Closed')
      and c.summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
      and c.is_target_population = true
      and c.dim_month >= date '{OBS_START}'
      and c.dim_month < date '{OBS_END}'
""")
print(f"Affected events: {len(affected_schools_query):,}")
print(affected_schools_query.head(10))

affected_schools_query.to_parquet(RAW_DIR / "affected_schools_for_verification.parquet", index=False)
print("saved")

In [ ]:
intercom_cols = describe_table("analytics_mart_support.rpt_intercom_conversations")
print(intercom_cols["column_name"].tolist())

In [ ]:
intercom_range = run_query("""
    select
          min(conversation_started_at) as earliest_conversation
        , max(conversation_started_at) as latest_conversation
        , count(*) as n_conversations
    from analytics_mart_support.rpt_intercom_conversations
""")
print(intercom_range)

In [ ]:
# Check whether transcript and CSAT fields are actually populated for OLDER conversations,
# specifically within your 2024+ training window, not just recent ones
historical_check = run_query("""
    select
          date_trunc('year', conversation_started_at) as year
        , count(*) as total_conversations
        , count(conversation_transcript) as has_transcript
        , count(csat_rating) as has_csat
        , count(teammate_customer_transcript) as has_teammate_transcript
    from analytics_mart_support.rpt_intercom_conversations
    where conversation_started_at >= '2024-01-01'
    group by 1
    order by 1
""")
print(historical_check)

In [ ]:
sample_conversations = run_query("""
    select
          conversation_id
        , school_id
        , conversation_started_at
        , csat_rating
        , csat_remark
        , is_fin_involved
        , is_human_resolved
        , conversation_transcript
    from analytics_mart_support.rpt_intercom_conversations
    where conversation_started_at >= '2024-01-01'
      and conversation_started_at < '2024-04-01'
      and conversation_transcript is not null
    order by conversation_started_at
    limit 5
""")

for i in range(len(sample_conversations)):
    row = sample_conversations.iloc[i]
    print(f"Conversation {row['conversation_id']} | school {row['school_id']} | {row['conversation_started_at']}")
    print(f"CSAT: {row['csat_rating']} | Fin involved: {row['is_fin_involved']} | Human resolved: {row['is_human_resolved']}")
    print(f"Transcript (first 500 chars): {str(row['conversation_transcript'])[:500]}")
    print("-" * 80)

In [ ]:
churner_school_ids = [78153, 196667, 197018, 201589, 206489, 98204, 220090, 99556, 104391, 225972,
                       107896, 231908, 246989, 128160, 253894, 135919, 143807, 29868, 205955, 206753]

churner_conversations = run_query(f"""
    select
          conversation_id
        , school_id
        , conversation_started_at
        , csat_rating
        , conversation_transcript
    from analytics_mart_support.rpt_intercom_conversations
    where school_id in ({','.join(map(str, churner_school_ids))})
      and conversation_started_at >= '2024-01-01'
      and conversation_transcript is not null
    order by school_id, conversation_started_at
    limit 15
""")

print(f"Found {len(churner_conversations)} conversations from these 20 actual churner schools")

for i in range(len(churner_conversations)):
    row = churner_conversations.iloc[i]
    print(f"\nSchool {row['school_id']} | {row['conversation_started_at']}")
    print(f"Transcript: {str(row['conversation_transcript'])[:400]}")
    print("-" * 80)

In [ ]:
# Check how often frustration/cancellation-intent words appear, comparing
# schools that actually churned vs schools that didn't, using a proper, larger sample

keyword_check = run_query("""
    with school_churn_status as (
        select
              school_id
            , max(case when is_reportable_request then 1 else 0 end) as ever_churned
        from analytics_mart_customer.rpt_school_churn_cases
        where dim_month >= '2024-01-01' and dim_month < '2026-01-01'
        group by school_id
    )
    select
          s.ever_churned
        , count(*) as total_conversations
        , sum(case when lower(c.conversation_transcript) like '%cancel%' then 1 else 0 end) as mentions_cancel
        , sum(case when lower(c.conversation_transcript) like '%frustrat%' then 1 else 0 end) as mentions_frustrated
        , sum(case when lower(c.conversation_transcript) like '%unhapp%' then 1 else 0 end) as mentions_unhappy
        , sum(case when lower(c.conversation_transcript) like '%disappoint%' then 1 else 0 end) as mentions_disappointed
        , sum(case when lower(c.conversation_transcript) like '%switch%' then 1 else 0 end) as mentions_switch
    from analytics_mart_support.rpt_intercom_conversations c
    join school_churn_status s on s.school_id = c.school_id
    where c.conversation_started_at >= '2024-01-01'
      and c.conversation_started_at < '2026-01-01'
      and c.conversation_transcript is not null
    group by s.ever_churned
""")
print(keyword_check)

In [ ]:
team_breakdown = run_query("""
    select
          team_name
        , team_group
        , count(*) as n_conversations
    from analytics_mart_support.rpt_intercom_conversations
    where conversation_started_at >= '2024-01-01'
    group by team_name, team_group
    order by n_conversations desc
    limit 30
""")
print(team_breakdown)

In [ ]:
cancel_team_conversations = run_query("""
    select
          conversation_id
        , school_id
        , conversation_started_at
        , team_name
        , conversation_transcript
    from analytics_mart_support.rpt_intercom_conversations
    where team_name in ('Subscriptions Cancel Requests', 'EEL Cancels')
      and conversation_started_at >= '2024-01-01'
      and conversation_transcript is not null
    order by conversation_started_at desc
    limit 10
""")

for i in range(len(cancel_team_conversations)):
    row = cancel_team_conversations.iloc[i]
    print(f"School {row['school_id']} | {row['team_name']} | {row['conversation_started_at']}")
    print(cancel_team_conversations.iloc[0]["conversation_transcript"])
    print("-" * 80)

In [ ]:
cancel_team_human = run_query("""
    select
          conversation_id
        , school_id
        , conversation_started_at
        , team_name
        , teammate_customer_transcript
    from analytics_mart_support.rpt_intercom_conversations
    where team_name in ('Subscriptions Cancel Requests', 'EEL Cancels')
      and conversation_started_at >= '2024-01-01'
      and teammate_customer_transcript is not null
    order by conversation_started_at desc
    limit 10
""")

for i in range(len(cancel_team_human)):
    row = cancel_team_human.iloc[i]
    print(f"School {row['school_id']} | {row['team_name']} | {row['conversation_started_at']}")
    print(f"Human transcript: {str(row['teammate_customer_transcript'])[:1500]}")
    print("-" * 80)

In [ ]:
full_transcripts = run_query("""
    select
          conversation_id
        , school_id
        , conversation_started_at
        , team_name
        , teammate_customer_transcript
        , length(teammate_customer_transcript) as transcript_length
    from analytics_mart_support.rpt_intercom_conversations
    where team_name in ('Subscriptions Cancel Requests', 'EEL Cancels')
      and conversation_started_at >= '2024-01-01'
      and teammate_customer_transcript is not null
    order by transcript_length desc
    limit 5
""")

for i in range(len(full_transcripts)):
    row = full_transcripts.iloc[i]
    print(f"School {row['school_id']} | {row['team_name']} | length: {row['transcript_length']} chars")
    print(row['teammate_customer_transcript'])
    print("=" * 100)

In [ ]:
one_school_id = 78153  # a confirmed churner from your earlier list, already had cancellation activity

one_school_all = run_query(f"""
    select
          conversation_id
        , conversation_started_at
        , team_name
        , conversation_transcript
        , teammate_customer_transcript
        , csat_rating
    from analytics_mart_support.rpt_intercom_conversations
    where school_id = {one_school_id}
      and (conversation_transcript is not null or teammate_customer_transcript is not null)
    order by conversation_started_at asc
""")

print(f"Total conversations for school {one_school_id}: {len(one_school_all)}")
for i in range(len(one_school_all)):
    row = one_school_all.iloc[i]
    print(f"\n[{i+1}] {row['conversation_started_at']} | Team: {row['team_name']} | CSAT: {row['csat_rating']}")
    content = row['teammate_customer_transcript'] if pd.notna(row['teammate_customer_transcript']) else row['conversation_transcript']
    print(str(content)[:2000])
    print("-" * 100)

In [ ]:
csat_check = run_query("""
    with school_churn_status as (
        select
              school_id
            , max(case when is_reportable_request then 1 else 0 end) as ever_churned
        from analytics_mart_customer.rpt_school_churn_cases
        where dim_month >= '2024-01-01' and dim_month < '2026-01-01'
        group by school_id
    )
    select
          s.ever_churned
        , count(c.csat_rating) as n_rated_conversations
        , avg(c.csat_rating) as avg_csat
        , sum(case when c.csat_rating <= 2 then 1 else 0 end) as n_low_csat
        , count(*) as total_conversations
    from analytics_mart_support.rpt_intercom_conversations c
    join school_churn_status s on s.school_id = c.school_id
    where c.conversation_started_at >= '2024-01-01'
      and c.conversation_started_at < '2026-01-01'
    group by s.ever_churned
""")
print(csat_check)

In [ ]:
team_churn_check = run_query("""
    with school_churn_status as (
        select
              school_id
            , max(case when is_reportable_request then 1 else 0 end) as ever_churned
        from analytics_mart_customer.rpt_school_churn_cases
        where dim_month >= '2024-01-01' and dim_month < '2026-01-01'
        group by school_id
    )
    select
          c.team_name
        , s.ever_churned
        , count(*) as n_conversations
        , count(distinct c.school_id) as n_schools
    from analytics_mart_support.rpt_intercom_conversations c
    join school_churn_status s on s.school_id = c.school_id
    where c.conversation_started_at >= '2024-01-01'
      and c.conversation_started_at < '2026-01-01'
    group by c.team_name, s.ever_churned
    order by c.team_name, s.ever_churned
""")
print(team_churn_check.to_string(index=False))

In [ ]:
# Count unique school + underlying event, not overlapping monthly labels
annual_churners = run_query("""
    select count(distinct school_id) as unique_schools
    from analytics_mart_customer.rpt_school_churn_cases
    where is_reportable_request = true
      and summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
      and cancellation_reason_category not in ('Failed Payment', 'Business Changes', 'Facility Closed')
      and is_target_population = true
      and dim_month >= '2025-01-01' and dim_month < '2026-01-01'
""")
print(annual_churners)

In [ ]:
case_type_check = run_query("""
    select
          cancellation_reason_category
        , summarized_request_type
        , count(*) as n
        , count(distinct school_id) as n_schools
    from analytics_mart_customer.rpt_school_churn_cases
    where cancellation_reason_category = 'Other'
       or resolved_cancellation_reason like '%no longer using%'
    group by cancellation_reason_category, summarized_request_type
    order by n desc
    limit 20
""")
print(case_type_check)

In [ ]:
# Find the actual, real source of "no longer using Brightwheel" as a case category,
# not a guess, search every text field for the exact phrase
source_check = run_query("""
    select
          resolved_status
        , resolved_status_cleaned
        , resolved_cancellation_reason
        , cancellation_reason_category
        , count(*) as n
    from analytics_mart_customer.rpt_school_churn_cases
    where lower(resolved_status) like '%no longer%'
       or lower(resolved_status_cleaned) like '%no longer%'
       or lower(resolved_cancellation_reason) like '%no longer%'
    group by 1, 2, 3, 4
    order by n desc
    limit 20
""")
print(source_check.to_string(index=False))


In [ ]:
missed_ids_unique = list(set([225972, 295126, 296660, 297893, 299632, 301480, 307523, 314122, 252376, 275662, 287636, 289225, 292236, 296658, 296659, 298716, 305325, 292940, 306078, 309195, 304826, 250269, 288657, 158300, 306723, 280631, 284576, 292950, 296742, 297389, 302311, 306574, 316673, 295141, 298268, 304447, 304957, 61282, 297876, 232675, 233659, 308107, 309380, 74027, 199538, 304841, 319249, 319805, 308473, 122282, 163629, 291018, 305806, 306047, 244819, 288753, 295156, 296633, 305578, 284863, 261351, 265009, 269200, 105587, 309985, 311802, 311824, 204090, 297875, 318052, 320168, 300314, 296519, 318750, 294998, 296652, 304279, 275088, 315986, 289738, 304932, 314331, 323312, 324401, 289838, 298554, 320276, 320826, 229444, 320889, 322248, 270081, 298722, 312003, 319089, 303328, 283598, 294968, 324322, 273874, 312230, 316516, 307586, 297578, 315612, 323154, 281897, 318970, 321775, 324613, 328180, 300117, 313652, 308339, 312316, 320360, 330591, 91627, 269034, 276057, 240945, 109939, 318365, 321567, 316649, 288032, 289039, 302445, 33743, 325903, 327489, 332997, 33030, 327166, 327395, 263387, 293844, 296745, 267328, 297468, 315197, 269053, 304463, 309723, 325120, 329131, 298798, 305911, 311050, 324806, 329132, 317704, 325304, 327787, 312834, 326636, 317183, 337601, 328188, 330593, 331249, 334074, 215814, 52438, 321544, 312661, 330978, 333417, 333758, 330051, 133377, 323503, 326693, 321422, 324603, 328123, 304541, 324248, 324261, 315789, 325993, 331029, 332165, 323850, 336746, 311842, 318745, 321946, 327155, 336449, 293855, 333305, 335556, 111307, 333662, 336732, 339195, 241181, 320003, 323221, 333068, 324960, 331053, 7274, 340202, 342, 340571, 318415, 325844, 319803, 340814, 281224, 302253, 104183, 320045, 333037, 336318, 324088, 326215, 339364, 338584, 279495, 290677, 296661, 300951, 308486, 308617, 294807, 299440, 305918, 310372, 312161, 297599, 301859, 303273, 306845, 311118, 314869, 288729, 136129, 307635, 102914, 56426, 299362, 314600, 284275, 296297, 304159, 293662, 296906, 308337, 271616, 311805, 318327, 311843, 316764, 298132, 311959, 260073, 302295, 275683, 297618, 309432, 310365, 304561, 305532, 308965, 60168, 307830, 311158, 307758, 292172, 82207, 308847, 309766, 313028, 154062, 312856, 313672, 316438, 202377, 310879, 319261, 304549, 316187, 324478, 227520, 312103, 303858, 308611, 320336, 287513, 304050, 304130, 301735, 303738, 318198, 318286, 320918, 284887, 324199, 35567, 47155, 250681, 309544, 306658, 307661, 310118, 318398, 319479, 322643, 294391, 301518, 310234, 276125, 307599, 309595, 316195, 328376, 297659, 315226, 94543, 320912, 322041, 321876, 320249, 298126, 299008, 310804, 319171, 325626, 315749, 322032, 320137, 307554, 317016, 324771, 297588, 312410, 327824, 321283, 202639, 304488, 324037, 327784, 303867, 309108, 310566, 294483, 263693, 315572, 323589, 326264, 316782, 322004, 316942, 332198, 312519, 316066, 318941, 309699, 324379, 331396, 150624, 293140, 306863, 318468, 321570, 271496, 322370, 334435, 338412, 328107, 315458, 318983, 321124, 326696, 322595, 331420, 319418, 320980, 323200, 324497, 327423, 336441, 337231, 334110, 316396, 308716, 308934, 310225, 312281, 327131, 335833, 305389, 308643, 22772, 300603, 328048, 331524, 323666, 330933, 321983, 184811, 340520, 191265, 250492, 333932, 182164, 320391, 326257, 335570, 338953, 339711, 337169, 337317, 341134, 319030, 321345, 323745, 325576, 340118, 333743, 325843, 334820, 335982, 308598, 320391, 326376, 340931, 340989, 319875, 326135, 270330, 307515, 321102, 338253, 340209]))

print(f"Unique missed school IDs: {len(missed_ids_unique)}")

In [ ]:
missed_ids_str = ','.join(map(str, missed_ids_unique))

csat_check_low_tenure = run_query(f"""
    select
          avg(c.csat_rating) as avg_csat
        , sum(case when c.csat_rating <= 2 then 1 else 0 end) as n_low_csat
        , count(c.csat_rating) as n_rated
        , count(*) as total_conversations
    from analytics_mart_support.rpt_intercom_conversations c
    where c.school_id in ({missed_ids_str})
""")
print(csat_check_low_tenure)

In [ ]:
csat_monthly = run_query("""
    select
          school_id
        , date_trunc('month', conversation_started_at)::date as dim_month
        , count(csat_rating) as n_rated
        , sum(case when csat_rating <= 2 then 1 else 0 end) as n_low_csat
    from analytics_mart_support.rpt_intercom_conversations
    where conversation_started_at >= '2024-01-01'
      and conversation_started_at < '2026-06-01'
      and school_id is not null
    group by school_id, date_trunc('month', conversation_started_at)
""")
print(f"Rows: {len(csat_monthly):,}")
csat_monthly.to_parquet(RAW_DIR / "csat_monthly.parquet", index=False)
print("saved")

In [ ]:
reasons = run_query("""
    select cancellation_reason_category, resolved_cancellation_reason, count(*) as n
    from analytics_mart_customer.rpt_school_churn_cases
    group by 1, 2
    order by n desc
""")

pattern = "dup|new account|merg|migrat|transfer|consolid|fresh|another account|second account|re-?open|start over"
hits = reasons[reasons["resolved_cancellation_reason"].str.contains(pattern, case=False, na=False)]
print("Migration-like reasons:")
print(hits.to_string(index=False))

print(f"\nAll reasons ({len(reasons)} total), top 60:")
print(reasons.head(60).to_string(index=False))

In [ ]:
wagner = run_query("""
    select *
    from analytics_mart_customer.mart_school_customers
    where lower(parse_name) like '%wagner%'
""")
print(wagner.T.to_string())

In [ ]:
check = run_query("""
    select
          coalesce(cancellation_reason_category, '(blank)') as category
        , coalesce(resolved_cancellation_reason, '(blank)') as reason
        , count(*) as n
    from analytics_mart_customer.rpt_school_churn_cases
    where coalesce(is_reportable_request, false)
      and summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
      and is_target_population
      and case_created_date is not null
      and dim_month >= '2024-01-01'
    group by 1, 2
    order by n desc
""")
total = check["n"].sum()
print(f"Label-eligible cancel cases: {total:,}")

blank = check.loc[check["category"] == "(blank)", "n"].sum()
print(f"With blank category: {blank:,} ({blank / total:.1%})")

not_real = check[(check["category"] == "Non SaaS Churn") |
                 check["reason"].isin(["Duplicate Case", "Not a True Cancel", "Made by Mistake",
                                       "Duplicate Account", "Transfer of Ownership"])]
print(f"\nNon-SaaS / duplicate / mistaken cases in the label: {not_real['n'].sum():,}")
print(not_real.to_string(index=False))

print("\nTop 40 overall:")
print(check.head(40).to_string(index=False))

In [ ]:
import json
from pathlib import Path

nb_path = next(Path("~/Proactive-Churn").expanduser().rglob("dataset_creation.ipynb"))
nb = json.load(open(nb_path))

for i, c in enumerate(nb["cells"]):
    src = "".join(c.get("source", []))
    for line in src.splitlines():
        if "permanent" in line.lower() or "closure" in line.lower():
            print(f"cell {i}: {line.strip()}")

In [ ]:
print("".join(nb["cells"][23]["source"]))

In [ ]:
and coalesce(resolved_cancellation_reason, '') <> 'Permanent Closure'

In [ ]:
hondo = run_query("""
    select dim_month, is_premium_sub_or_invoice, product_active_students, customer_segment
    from analytics_mart_customer.rpt_monthly_school_usage_trends
    where school_id = 211562
      and dim_month >= '2025-08-01'
    order by dim_month
""")
print(hondo.to_string(index=False))

hondo_cases = run_query("""
    select case_created_date, summarized_request_type, resolved_status,
           cancellation_reason_category, resolved_cancellation_reason
    from analytics_mart_customer.rpt_school_churn_cases
    where school_id = 211562
    order by case_created_date
""")
print(hondo_cases.to_string(index=False))

In [ ]:
ghosts = run_query("""
    with last_cancel as (
        select school_id, max(case_created_date) as cancel_date
        from analytics_mart_customer.rpt_school_churn_cases
        where resolved_status = 'Canceled'
          and summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
        group by school_id
    )
    select count(*) as ghost_rows, count(distinct t.school_id) as ghost_schools
    from analytics_mart_customer.rpt_monthly_school_usage_trends t
    join last_cancel c on c.school_id = t.school_id
    where t.is_premium_sub_or_invoice
      and not t.is_gov
      and t.customer_segment in ('10-39 small', '40-99 medium', '100+ large', 'SMB Multisite 2-9')
      and upper(trim(t.school_country)) in ('UNITED STATES', 'CANADA')
      and t.dim_month >= dateadd(month, 2, date_trunc('month', c.cancel_date))
      and coalesce(t.product_active_students, 0) = 0
      and t.dim_month >= '2024-01-01'
      and t.dim_month < '2026-06-01'
""")
print(ghosts)
print(f"Share of the 800,414-row training extract: {ghosts['ghost_rows'].iloc[0] / 800414:.1%}")

In [ ]:
LIST_IDS = "301056,265509,119790,312661,268984,225615,261114,43758,198787,253022,313414,337564,169313,290243,270231,233483,294470,326264,269067,248707,316438,269573,155540,219679,321775,223358,206353,156922,242474,271263,303406,226036,260099,146157,232458,264954,321567,92344,295690,248394,304461,155559,260849,180113,235123,263902,250329,284364,246428,247106,237943,287636,268306,186863,259717,153949,3116,32225,249662,230967,275088,98647,94307,221533,299251,94915,141692,227520,164349,237286,298988,282261,315730,231297,294474,190125,17949,302268,252719,196186,220416,226331,318745,260116,167992,311156,227185,274478,333036,296109,273058,73273,292172,255563,273425,311662,1087,159408,279276,119720,139597,309595,241055,198385,119621,201108,289341,292672,220627,229396,236874,302036,290403,171600,334435,324183,332827,225918,283710,164095,250188,262808,250620,332055,286942,287326,175055,188094,200887,171934,225513,245722,185805,247786,309511,2187,202663,275369,208962,297468,326146,279573,227183,146304,241692,296791,161196,245947,292188,206050,188506,33030,323374,328111,270052,292114,288674,226945,185498,291215,278635,266368,330692,232652,305752,284863,255149,288906,284287,315687,323693,304927,183663,288626,308204,313223,316656,315952,315027,293716,226529,308681,210260,189994,232538,287895,211562,169582,211464,178061,269243,157940,277110,220350,74268,280969,316673,312513,271415,87651"

list_ghosts = run_query(f"""
    with last_cancel as (
        select school_id, max(case_created_date) as cancel_date
        from analytics_mart_customer.rpt_school_churn_cases
        where resolved_status = 'Canceled'
          and summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
          and case_created_date < '2026-06-01'
        group by school_id
    )
    select
          t.school_id
        , c.cancel_date
        , t.product_active_students as active_students_may
        , t.is_premium_sub_or_invoice as premium_flag_may
        , case when t.dim_month >= dateadd(month, 2, date_trunc('month', c.cancel_date))
                and coalesce(t.product_active_students, 0) = 0
               then 'GHOST' else 'cancelled earlier, still active or paid-through' end as status
    from analytics_mart_customer.rpt_monthly_school_usage_trends t
    join last_cancel c on c.school_id = t.school_id
    where t.dim_month = '2026-05-01'
      and t.school_id in ({LIST_IDS})
    order by status, c.cancel_date
""")

print(f"Schools on the list with a cancelled SaaS case before scoring: {len(list_ghosts)}")
print(f"Of those, ghosts (2+ months after cancelling, zero active students): {(list_ghosts['status'] == 'GHOST').sum()}")
print(list_ghosts.to_string(index=False))

In [ ]:
CANC_IDS = "265509,268984,261114,290243,233483,326264,219679,223358,271263,92344,248394,246428,237943,259717,249662,275088,98647,252719,220416,318745,167992,333036,273058,292172,241055,290403,334435,332827,250188,250620,225513,185805,146304,33030,292114,226945,291215,284863,315687,313223,293716,308681,189994,211562,277110"

cases = run_query(f"""
    select school_id, case_created_date, resolved_status,
           cancellation_reason_category, resolved_cancellation_reason
    from analytics_mart_customer.rpt_school_churn_cases
    where school_id in ({CANC_IDS})
      and summarized_request_type in ('Addressable Cancelation Request', 'Non-Addressable Cancelation Request')
      and case_created_date >= '2025-01-01'
    order by school_id, case_created_date
""")

# Most recent cancel request per school
latest = cases.sort_values("case_created_date").groupby("school_id").tail(1).copy()
latest["month"] = pd.to_datetime(latest["case_created_date"]).dt.to_period("M").astype(str)

def when(m):
    if m < "2026-05": return "1. before May snapshot"
    if m == "2026-05": return "2. May 2026"
    if m <= "2026-08": return "3. Jun-Aug 2026 (3-month window)"
    return "4. Sep-Oct 2026 (after window)"
latest["when"] = latest["month"].map(when)

found = set(latest["school_id"])
missing = [i for i in map(int, CANC_IDS.split(",")) if i not in found]
print(latest["when"].value_counts().sort_index().to_string())
print(f"\nNo cancel request found since 2025: {len(missing)} schools: {missing}")
print(latest[["school_id", "month", "when", "resolved_status", "resolved_cancellation_reason"]].to_string(index=False))

In [ ]:
# 1. Is there a field that marks test or internal accounts?
cols = run_query("""
    select column_name
    from information_schema.columns
    where table_schema = 'analytics_mart_customer'
      and table_name = 'mart_school_customers'
""")["column_name"]
print(cols[cols.str.contains("test|demo|internal|fake|sandbox", case=False)].tolist())
# 2. Everything we know about this school
abz = run_query("""
    select * from analytics_mart_customer.mart_school_customers
    where school_id = 282261
""")
print(abz.T.to_string())

# 3. Its monthly history
abz_hist = run_query("""
    select dim_month, is_premium_sub_or_invoice, product_active_students, customer_segment
    from analytics_mart_customer.rpt_monthly_school_usage_trends
    where school_id = 282261 and dim_month >= '2025-01-01'
    order by dim_month
""")
print(abz_hist.to_string(index=False))

# 4. How many scored schools have "test" or "demo" in the name?
test_like = run_query("""
    select school_id, parse_name
    from analytics_mart_customer.mart_school_customers
    where lower(parse_name) similar to '%(test|demo|sandbox)%'
""")
print(f"Schools with test-like names: {len(test_like)}")
print(test_like.head(30).to_string(index=False))

In [ ]:
test_in_pop = run_query("""
    select t.school_id, c.parse_name, c.is_current_saas_subscription_test, c.is_fraud
    from analytics_mart_customer.rpt_monthly_school_usage_trends t
    join analytics_mart_customer.mart_school_customers c on c.school_id = t.school_id
    where t.dim_month = '2026-09-01'
      and t.is_premium_sub_or_invoice
      and not t.is_gov
      and t.customer_segment in ('10-39 small', '40-99 medium', '100+ large', 'SMB Multisite 2-9')
      and upper(trim(t.school_country)) in ('UNITED STATES', 'CANADA')
      and (lower(c.parse_name) similar to '%(test|demo|sandbox)%'
           or c.is_current_saas_subscription_test
           or c.is_fraud)
""")
print(f"Possible test or fraud accounts in the scored population: {len(test_in_pop)}")
print(test_in_pop.to_string(index=False))

In [ ]:
SCORE_MONTH = str(run_query(f"""
    select max(dim_month) as m from {TABLES['usage_trends']}
    where dim_month < date_trunc('month', current_date)
""").iloc[0, 0])
print("Scoring month:", SCORE_MONTH)

SCORING_SQL = f"""
with obs as (
    select {_sel_obs}
    from {TABLES['usage_trends']}
    where dim_month = date '{SCORE_MONTH}'
      and {PREMIUM_FILTER}
      and {GOV_FILTER}
      and customer_segment in ({_seg_ph})
      and upper(trim(school_country)) in ({_ctry_ph})
)
, billing as (
    select school_id, dim_month,
           months_since_first_processed_dollars_local, processed_dollars_rank_local,
           has_processed_dollars_this_month_local, has_processed_dollars_last_month_local,
           monthly_billing_status_local
    from {TABLES['billing_history']}
    where dim_month = date '{SCORE_MONTH}'
)
select o.*,
       b.months_since_first_processed_dollars_local, b.processed_dollars_rank_local,
       b.has_processed_dollars_this_month_local, b.has_processed_dollars_last_month_local,
       b.monthly_billing_status_local,
       case when b.months_since_first_processed_dollars_local is not null then 1 else 0 end as has_ever_billed
from obs o
left join billing b on b.school_id = o.school_id and b.dim_month = o.dim_month
"""
score_raw = run_query(SCORING_SQL, params=tuple(POPULATION_SEGMENTS) + tuple(POPULATION_COUNTRIES))
print(f"{len(score_raw):,} schools | has_ever_billed rate {score_raw['has_ever_billed'].mean():.1%} (training ~77%)")

last_active = run_query(f"""
    select school_id, max(dim_month) as last_active_month
    from {TABLES['usage_trends']}
    where product_active_students > 0 and dim_month <= date '{SCORE_MONTH}'
    group by school_id
""")

_type_ph = ", ".join(["%s"] * len(CANCEL_REQUEST_TYPES))
cases = run_query(f"""
    select school_id,
           max(case when resolved_status = 'Canceled' then case_created_date end) as last_completed_cancel,
           max(case_created_date) as last_cancel_request
    from {TABLES['churn_cases']}
    where summarized_request_type in ({_type_ph})
    group by school_id
""", params=tuple(CANCEL_REQUEST_TYPES))

cust = run_query("""
    select school_id, brightwheel_school_uuid, parse_name as account_name, has_open_churn_case,
           is_current_saas_subscription_premium, is_current_saas_subscription_canceled
    from analytics_mart_customer.mart_school_customers
""")

gpv = run_query(f"""
    select school_id, avg(billing_net_gpv_local) as avg_monthly_gpv_nonzero
    from {TABLES['usage_trends']}
    where billing_net_gpv_local > 0
      and dim_month >  dateadd(month, -12, date '{SCORE_MONTH}')
      and dim_month <= date '{SCORE_MONTH}'
    group by school_id
""")

for name, df in [("score_raw", score_raw), ("last_active", last_active),
                 ("cases", cases), ("cust", cust), ("gpv", gpv)]:
    df.to_parquet(RAW_DIR / f"newlist_{name}.parquet", index=False)
print("saved")

In [ ]:
final = pd.read_csv(RAW_DIR / "v3_list_2026_09.csv")


In [ ]:
ids = ",".join(map(str, final["School ID"]))
reach = run_query(f"""
    select school_id, parse_name, last_mobile_or_web_activity_date,
           school_owner_email, school_owner_phone_number, primary_contact_phone, fallback_contact_phone
    from analytics_mart_customer.mart_school_customers
    where school_id in ({ids})
""")
reach["days_since_login"] = (pd.Timestamp.today() - pd.to_datetime(reach["last_mobile_or_web_activity_date"])).dt.days
has_phone = reach[["school_owner_phone_number", "primary_contact_phone", "fallback_contact_phone"]].notna().any(axis=1)

print(f"Logged in within 30 days: {(reach['days_since_login'] <= 30).sum()} of {len(reach)}")
print(f"No login in 60+ days: {(reach['days_since_login'] > 60).sum()}")
print(f"Have a phone number: {has_phone.sum()} | Have an owner email: {reach['school_owner_email'].notna().sum()}")
print(reach.loc[reach["days_since_login"] > 60, ["school_id", "parse_name", "days_since_login"]].to_string(index=False))

In [ ]:
final = pd.read_csv(RAW_DIR / "v3_list_2026_09.csv")
high = final[final["Avg monthly GPV (non-zero months)"] >= 10000][["School ID", "Account Name"]]
high_ids = ",".join(map(str, high["School ID"]))

recent = run_query(f"""
    select school_id, dim_month, billing_net_gpv_local
    from {TABLES['usage_trends']}
    where school_id in ({high_ids})
      and dim_month between date '2026-06-01' and date '2026-09-01'
""")
trend = recent.pivot(index="school_id", columns="dim_month", values="billing_net_gpv_local").round(0)
trend.columns = [pd.to_datetime(c).strftime("%b") for c in trend.columns]
trend = trend.merge(high, left_index=True, right_on="School ID").set_index("Account Name").drop(columns="School ID")
print(trend.to_string())

In [ ]:
final = pd.read_csv(RAW_DIR / "v3_list_2026_09.csv")
ids = ",".join(map(str, final["School ID"]))
_type_ph = ", ".join(["%s"] * len(CANCEL_REQUEST_TYPES))

last_case = run_query(f"""
    select school_id, case_created_date, resolved_status, resolved_cancellation_reason
    from (
        select school_id, case_created_date, resolved_status, resolved_cancellation_reason,
               row_number() over (partition by school_id order by case_created_date desc) as rn
        from {TABLES['churn_cases']}
        where school_id in ({ids})
          and summarized_request_type in ({_type_ph})
    ) t
    where rn = 1
""", params=tuple(CANCEL_REQUEST_TYPES))

print(f"Schools on the list with a past cancel request: {len(last_case)} of {len(final)}")
print(last_case["resolved_status"].value_counts().to_string())
print(last_case.sort_values("case_created_date", ascending=False).head(20).to_string(index=False))

In [ ]:
CONTROL_IDS = "312661,225615,198787,313414,169313,270231,294470,248707,269573,321775,206353,242474,303406,260099,232458,321567,295690,304461,180113,263902,284364,247106,287636,186863,153949,32225,230967,94307,299251,141692,164349,298988,315730,294474,302268,196186,226331,260116,311156,274478,296109,73273,255563,311662,159408,119720,309595,198385,201108,292672,229396,302036,171600,324183,225918,164095,262808,332055,287326,188094,171934,245722,309511,202663,208962,326146,227183,241692,161196,292188,188506,323374,270052,288674,185498,278635,330692,305752,255149,284287,323693,183663,308204,316656,315027,226529,210260,232538,169582,178061,157940,220350,280969,312513,87651"
_type_ph = ", ".join(["%s"] * len(CANCEL_REQUEST_TYPES))

req = run_query(f"""
    select school_id, min(case_created_date) as first_request, max(resolved_status) as status
    from {TABLES['churn_cases']}
    where school_id in ({CONTROL_IDS})
      and summarized_request_type in ({_type_ph})
      and coalesce(is_reportable_request, false)
      and case_created_date >= '2026-06-01' and case_created_date < '2026-09-01'
    group by school_id
""", params=tuple(CANCEL_REQUEST_TYPES))

print(f"Control schools with a cancel request in June–August: {len(req)} of 95 ({len(req) / 95:.0%})")
print(req["status"].value_counts().to_string())

In [ ]:
def control_requests(start, end):
    r = run_query(f"""
        select school_id, max(resolved_status) as status
        from {TABLES['churn_cases']}
        where school_id in ({CONTROL_IDS})
          and summarized_request_type in ({_type_ph})
          and coalesce(is_reportable_request, false)
          and case_created_date >= '{start}' and case_created_date < '{end}'
        group by school_id
    """, params=tuple(CANCEL_REQUEST_TYPES))
    return r

jun_aug = control_requests("2026-06-01", "2026-09-01")
sep_oct = control_requests("2026-09-01", "2026-11-01")
jun_oct = control_requests("2026-06-01", "2026-11-01")

for label, r in [("June–August", jun_aug), ("September–October", sep_oct), ("June–October, any request", jun_oct)]:
    print(f"{label}: {len(r)} of 95 ({len(r) / 95:.0%}) | {r['status'].value_counts().to_dict()}")

new_in_sep_oct = set(sep_oct["school_id"]) - set(jun_aug["school_id"])
print(f"Schools whose first request in this period came in September–October: {len(new_in_sep_oct)}")

In [ ]:
SCORE_MONTH = str(run_query(f"""
    select max(dim_month) as m from {TABLES['usage_trends']}
    where dim_month < date_trunc('month', current_date)
""").iloc[0, 0])
print("Scoring month:", SCORE_MONTH, "(expect 2026-10-01)")

SCORING_SQL = f"""
with obs as (
    select {_sel_obs}
    from {TABLES['usage_trends']}
    where dim_month = date '{SCORE_MONTH}'
      and {PREMIUM_FILTER}
      and {GOV_FILTER}
      and customer_segment in ({_seg_ph})
      and upper(trim(school_country)) in ({_ctry_ph})
)
, billing as (
    select school_id, dim_month,
           months_since_first_processed_dollars_local, processed_dollars_rank_local,
           has_processed_dollars_this_month_local, has_processed_dollars_last_month_local,
           monthly_billing_status_local
    from {TABLES['billing_history']}
    where dim_month = date '{SCORE_MONTH}'
)
select o.*,
       b.months_since_first_processed_dollars_local, b.processed_dollars_rank_local,
       b.has_processed_dollars_this_month_local, b.has_processed_dollars_last_month_local,
       b.monthly_billing_status_local,
       case when b.months_since_first_processed_dollars_local is not null then 1 else 0 end as has_ever_billed
from obs o
left join billing b on b.school_id = o.school_id and b.dim_month = o.dim_month
"""
score_raw = run_query(SCORING_SQL, params=tuple(POPULATION_SEGMENTS) + tuple(POPULATION_COUNTRIES))
print(f"{len(score_raw):,} schools | has_ever_billed rate {score_raw['has_ever_billed'].mean():.1%} (expect ~77%)")

last_active = run_query(f"""
    select school_id, max(dim_month) as last_active_month
    from {TABLES['usage_trends']}
    where product_active_students > 0 and dim_month <= date '{SCORE_MONTH}'
    group by school_id
""")

_type_ph = ", ".join(["%s"] * len(CANCEL_REQUEST_TYPES))
cases = run_query(f"""
    select school_id,
           max(case when resolved_status = 'Canceled' then case_created_date end) as last_completed_cancel,
           max(case_created_date) as last_cancel_request
    from {TABLES['churn_cases']}
    where summarized_request_type in ({_type_ph})
    group by school_id
""", params=tuple(CANCEL_REQUEST_TYPES))

cust = run_query("""
    select school_id, brightwheel_school_uuid, parse_name as account_name, has_open_churn_case,
           is_current_saas_subscription_premium, is_current_saas_subscription_canceled
    from analytics_mart_customer.mart_school_customers
""")

gpv = run_query(f"""
    select school_id, avg(billing_net_gpv_local) as avg_monthly_gpv_nonzero
    from {TABLES['usage_trends']}
    where billing_net_gpv_local > 0
      and dim_month >  dateadd(month, -12, date '{SCORE_MONTH}')
      and dim_month <= date '{SCORE_MONTH}'
    group by school_id
""")

for name, df in [("score_raw", score_raw), ("last_active", last_active),
                 ("cases", cases), ("cust", cust), ("gpv", gpv)]:
    df.to_parquet(RAW_DIR / f"newlist_{name}.parquet", index=False)
print("saved")

In [ ]:
final = pd.read_csv(RAW_DIR / "v3_list_2026_10.csv")
ids = ",".join(map(str, final["School ID"]))

reach = run_query(f"""
    select school_id, parse_name, last_mobile_or_web_activity_date,
           school_owner_email, school_owner_phone_number, primary_contact_phone, fallback_contact_phone
    from analytics_mart_customer.mart_school_customers
    where school_id in ({ids})
""")
reach["days_since_login"] = (pd.Timestamp.today() - pd.to_datetime(reach["last_mobile_or_web_activity_date"])).dt.days
has_phone = reach[["school_owner_phone_number", "primary_contact_phone", "fallback_contact_phone"]].notna().any(axis=1)

print(f"Logged in within 30 days: {(reach['days_since_login'] <= 30).sum()} of {len(reach)}")
print(f"No login in 60+ days: {(reach['days_since_login'] > 60).sum()}")
print(f"Have a phone number: {has_phone.sum()} | Have an owner email: {reach['school_owner_email'].notna().sum()}")
print(reach.loc[reach["days_since_login"] > 60, ["school_id", "parse_name", "days_since_login"]]
      .sort_values("days_since_login", ascending=False).to_string(index=False))

In [ ]:
high = final[final["Avg monthly GPV (non-zero months)"] >= 10000][["School ID", "Account Name"]]
high_ids = ",".join(map(str, high["School ID"]))

recent = run_query(f"""
    select school_id, dim_month, billing_net_gpv_local
    from {TABLES['usage_trends']}
    where school_id in ({high_ids})
      and dim_month between date '2026-07-01' and date '2026-10-01'
""")
trend = recent.pivot(index="school_id", columns="dim_month", values="billing_net_gpv_local").round(0)
trend.columns = [pd.to_datetime(c).strftime("%b") for c in trend.columns]
trend = trend.merge(high, left_index=True, right_on="School ID").set_index("Account Name").drop(columns="School ID")

def group(r):
    if r.get("Oct", 0) >= 0.5 * max(r.get("Sep", 0), 1):
        return "Still billing"
    if r.get("Sep", 0) > 0 and r.get("Oct", 0) < 0.1 * r.get("Sep", 1):
        return "Stopped suddenly in October"
    return "Stopped earlier"
trend["Group"] = trend.apply(group, axis=1)

print(f"High-value schools ($10k+/month): {len(trend)}")
print(trend["Group"].value_counts().to_string())
print(trend.sort_values("Group").to_string())

In [ ]:
logins = run_query("""
    select school_id, last_mobile_or_web_activity_date
    from analytics_mart_customer.mart_school_customers
""")
logins.to_parquet(RAW_DIR / "newlist_logins.parquet", index=False)

In [ ]:
kidzone = run_query("""
    select school_id, parse_name, organization_id, school_owner_email, school_created_date,
           is_current_saas_subscription_premium, last_mobile_or_web_activity_date, active_student_count
    from analytics_mart_customer.mart_school_customers
    where school_owner_email = (select school_owner_email from analytics_mart_customer.mart_school_customers where school_id = 167711)
       or organization_id = (select organization_id from analytics_mart_customer.mart_school_customers where school_id = 167711)
    order by school_created_date
""")
print(kidzone.to_string(index=False))